# RCT data preparation

This notebook prepares the RayCloudTools (RCT) data used in the final dissertation analysis.

The original RCT outputs, GT matching tables and manual QC records are kept unchanged. This notebook combines these files into consistent analysis-ready tables for the Easy, Intermediate and Challenging terrains.

The main outputs are:

- a candidate-level table for the RCT tree-detection analysis;
- a paired before/after QC table for TH and DBH;
- checks against previously generated results to make sure the cleaned workflow reproduces the final dissertation analysis.

`segment_id` is used throughout as the canonical RCT tree identifier. The older `RCT_tree_id` field was derived as `segment_id + 1` and is not used in the final cleaned analysis.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import re



# Set up the project folders
# All file paths are defined relative to the main dissertation-analysis folder 
# This makes the notebook easier to reproduce if the project is moved or downloaded from GitHub.


def find_project_root(start_path):
    """
    Find the main dissertation-analysis folder.

    The project root is identified by checking for both the
    'data' and 'notebooks' folders.
    """

    start_path = Path(start_path).resolve()
    
    # Start from the current folder and move upwards until
    # the main project folder is found.
    for folder in [start_path, *start_path.parents]:

        if (
            (folder / "data").is_dir()
            and (folder / "notebooks").is_dir()
        ):
            return folder
    # Stop with a clear error if the expected project folder structure cannot be found
    raise FileNotFoundError(
        "Could not find the project root. "
        "Expected a folder containing both 'data' and 'notebooks'."
    )

# Find the project root from the notebook's current location
PROJECT_DIR = find_project_root(Path.cwd())

# Main data folder
DATA_DIR = PROJECT_DIR / "data"

# Input, manual QC, validation and processed-data folders
RAW_RCT_DIR = DATA_DIR / "raw" / "rct"
MANUAL_QC_DIR = DATA_DIR / "manual_qc" / "rct"
VALIDATION_DIR = DATA_DIR / "validation" / "rct"
PROCESSED_DIR = DATA_DIR / "processed"

# Separate raw RCT folders for each terrain
EASY_DIR = RAW_RCT_DIR / "easy"
INTERMEDIATE_DIR = RAW_RCT_DIR / "intermediate"
CHALLENGING_DIR = RAW_RCT_DIR / "challenging"

# Check that the project folder was found correctly:
print(f"Project root found: {PROJECT_DIR.name}")

Project root found: dissertation-analysis


In [2]:
# Check the required project folders
# Check that all folders needed by this notebook are present before any data are loaded.

required_folders = {
    "Easy RCT data": EASY_DIR,
    "Intermediate RCT data": INTERMEDIATE_DIR,
    "Challenging RCT data": CHALLENGING_DIR,
    "Manual QC": MANUAL_QC_DIR,
    "Validation data": VALIDATION_DIR,
    "Processed outputs": PROCESSED_DIR,
}


for name, path in required_folders.items():
    
    # Report whether each expected folder can be found
    if path.exists():
        print(f"OK      {name}: {path.relative_to(PROJECT_DIR)}")

    else:
        print(f"MISSING {name}: {path.relative_to(PROJECT_DIR)}")

OK      Easy RCT data: data/raw/rct/easy
OK      Intermediate RCT data: data/raw/rct/intermediate
OK      Challenging RCT data: data/raw/rct/challenging
OK      Manual QC: data/manual_qc/rct
OK      Validation data: data/validation/rct
OK      Processed outputs: data/processed


In [3]:
# Define consistent terrain, scanner and RCT labels
# Some older files use different spellings or names for the same scanner
# They are converted to one standard set of labels so that each scanner is treated as a single category 

TERRAINS = [
    "Easy",
    "Intermediate",
    "Challenging",
]


SCANNERS = [
    "Single sensor",
    "Dual sensor",
    "Hovermap",
]

# Map older scanner names and spelling variants onto the
# standard labels used throughout the notebook.
SCANNER_NAME_MAP = {
    "Jednoskenner": "Single sensor",
    "Jednoskener": "Single sensor",
    "Single sensor": "Single sensor",

    "Dvojskenner": "Dual sensor",
    "Dvojskener": "Dual sensor",
    "Dual sensor": "Dual sensor",

    "Hovermap": "Hovermap",
}



# Set the RCT tree identifier used throughout the analysis
# 'segment_id' is the identifier assigned directly to each tree segment by RCT
# Some older files also contain 'RCT_tree_id', which was created as segment_id + 1 so that numbering started from 1
# segment_id is used as the single consistent RCT identifier here.

RCT_ID_COLUMN = "segment_id"

# Print the main labels used by the notebook as a quick check:
print("Terrains:", TERRAINS)
print("Scanners:", SCANNERS)
print("Canonical RCT ID column:", RCT_ID_COLUMN)

Terrains: ['Easy', 'Intermediate', 'Challenging']
Scanners: ['Single sensor', 'Dual sensor', 'Hovermap']
Canonical RCT ID column: segment_id


In [4]:
# Define the Easy-terrain input files
# The source files are listed explicitly rather than searched for automatically

EASY_FILES = {
    "GT matches":
        EASY_DIR / "easy_GT_to_RCT_matches.csv",

    "Candidate QC":
        EASY_DIR / "easy_RCT_candidate_QC.csv",

    "Single sensor original attributes":
        EASY_DIR / "et_j_nogrid_tree_attributes.csv",

    "Dual sensor original attributes":
        EASY_DIR / "et_dvojskener_nogrid_tree_attributes.csv",

    "Hovermap original attributes":
        EASY_DIR / "et_hovermap_nogrid_tree_attributes.csv",

    "Single sensor final attributes":
        EASY_DIR / "easy_jednoskenner_final_attribute_matches.csv",

    "Dual sensor final attributes":
        EASY_DIR / "easy_dvojskenner_final_attribute_matches.csv",

    "Hovermap final attributes":
        EASY_DIR / "easy_hovermap_final_attribute_matches.csv",

    "Manual QC decisions":
        MANUAL_QC_DIR / "easy" / "easy_manual_QC_decisions.csv",
}

# Check that every Easy input file is available before loading it

missing_easy_files = []

for name, path in EASY_FILES.items():

    # Report whether each expected file can be found
    if path.exists():
        print(f"OK      {name}")

    else:
        print(f"MISSING {name}")
        missing_easy_files.append(path)

# Stop here if any required input is missing
if missing_easy_files:
    raise FileNotFoundError(
        "One or more Easy-terrain input files are missing. "
        "Check the folder structure before continuing."
    )

OK      GT matches
OK      Candidate QC
OK      Single sensor original attributes
OK      Dual sensor original attributes
OK      Hovermap original attributes
OK      Single sensor final attributes
OK      Dual sensor final attributes
OK      Hovermap final attributes
OK      Manual QC decisions


In [5]:
# Load the Easy-terrain input files
# Each source file is loaded into its own pandas DataFrame.
# At this stage the files are only being read and inspected;
# the original CSV files are not modified.


easy_gt_matches = pd.read_csv(EASY_FILES["GT matches"])
easy_candidate_qc = pd.read_csv(EASY_FILES["Candidate QC"])

easy_single_original = pd.read_csv(
    EASY_FILES["Single sensor original attributes"]
)

easy_dual_original = pd.read_csv(
    EASY_FILES["Dual sensor original attributes"]
)

easy_hovermap_original = pd.read_csv(
    EASY_FILES["Hovermap original attributes"]
)

easy_single_final = pd.read_csv(
    EASY_FILES["Single sensor final attributes"]
)

easy_dual_final = pd.read_csv(
    EASY_FILES["Dual sensor final attributes"]
)

easy_hovermap_final = pd.read_csv(
    EASY_FILES["Hovermap final attributes"]
)

easy_manual_qc = pd.read_csv(
    EASY_FILES["Manual QC decisions"]
)

# Group the loaded tables together so they can all be checked using the same inspection code

easy_tables = {
    "GT matches": easy_gt_matches,
    "Candidate QC": easy_candidate_qc,
    "Single sensor original attributes": easy_single_original,
    "Dual sensor original attributes": easy_dual_original,
    "Hovermap original attributes": easy_hovermap_original,
    "Single sensor final attributes": easy_single_final,
    "Dual sensor final attributes": easy_dual_final,
    "Hovermap final attributes": easy_hovermap_final,
    "Manual QC decisions": easy_manual_qc,
}

# Check the size and column structure of each input table
# To confirm that the expected files and columns have been loaded before any matching or cleaning is carried out
for name, df in easy_tables.items():

    print("\n" + "=" * 70)
    print(name)
    print(f"Rows: {len(df)}")
    print(f"Columns: {len(df.columns)}")
    print("Column names:")
    print(df.columns.tolist())


GT matches
Rows: 27
Columns: 22
Column names:
['dataset', 'GT_ID', 'GT_X', 'GT_Y', 'GT_TH', 'GT_DBH', 'RCT_tree_id', 'segment_id', 'RCT_X', 'RCT_Y', 'RCT_height', 'RCT_DBH', 'crown_radius', 'total_vol_L', 'match_distance_m', 'nearest_candidate_distance_m', 'second_nearest_candidate_distance_m', 'ambiguity_gap_m', 'assigned_is_nearest', 'review_status', 'review_reason', 'source_candidate_csv']

Candidate QC
Rows: 31
Columns: 15
Column names:
['dataset', 'RCT_tree_id', 'segment_id', 'RCT_X', 'RCT_Y', 'RCT_height', 'RCT_DBH', 'crown_radius', 'total_vol_L', 'candidate_status', 'assigned_GT_ID', 'assigned_distance_m', 'nearest_GT_ID', 'nearest_GT_distance_m', 'source_candidate_csv']

Single sensor original attributes
Rows: 10
Columns: 14
Column names:
['dataset', 'tree_id', 'segment_id', 'height', 'crown_radius', 'dimension', 'monocotal', 'DBH', 'bend', 'branch_slope', 'x', 'y', 'z', 'total_vol_L']

Dual sensor original attributes
Rows: 10
Columns: 14
Column names:
['dataset', 'tree_id', '

In [6]:
# Check the integrity of the Easy-terrain data
# Before combining the Easy-terrain tables, check that:
#1. the expected records are present
#2. the tree IDs are consistent
#3. and all final RCT segments can be traced back to the original candidate data
# These checks stop the notebook immediately if an unexpected ID or record-count problem is found

def require(condition, message):
    """Stop the notebook if an expected data condition is not met."""

    if not condition:
        raise AssertionError(message)


# First inspect the dataset labels stored in the source files
# Check how many candidate segments are stored under each scanner dataset label before carrying out any matching
print("Dataset labels in Easy candidate table:")
print(easy_candidate_qc["dataset"].value_counts())
print()


# Expected numbers of records
# These counts are based on the final Easy-terrain files used in the analysis 
# They provide a quick check that the correct versions of the input files have been loaded
require(
    len(easy_candidate_qc) == 31,
    "Expected 31 Easy RCT candidates."
)

require(
    len(easy_gt_matches) == 27,
    "Expected 27 Easy GT-to-RCT match records."
)

require(
    len(easy_single_final) == 9,
    "Expected 9 final Single-sensor matches."
)

require(
    len(easy_dual_final) == 9,
    "Expected 9 final Dual-sensor matches."
)

require(
    len(easy_hovermap_final) == 9,
    "Expected 9 final Hovermap matches."
)



# Check that segment_id is unique within each RCT dataset

# segment_id only needs to be unique within each scanner dataset, because the same number can occur in different RCT datasets
duplicate_candidates = easy_candidate_qc.duplicated(
    subset=["dataset", "segment_id"]
).sum()

require(
    duplicate_candidates == 0,
    "Duplicate dataset + segment_id combinations found in Candidate QC."
)

# Check the original attribute tables separately to ensure that each RCT segment appears only once within each scanner
for name, df in {
    "Single sensor original": easy_single_original,
    "Dual sensor original": easy_dual_original,
    "Hovermap original": easy_hovermap_original,
}.items():

    require(
        df["segment_id"].is_unique,
        f"Duplicate segment_id values found in {name} attributes."
    )


# 4. Check the GT-to-RCT matching structure
# A single GT tree should not be matched to more than one RCT segment within the same scanner dataset
duplicate_gt_matches = easy_gt_matches.duplicated(
    subset=["dataset", "GT_ID"]
).sum()

require(
    duplicate_gt_matches == 0,
    "A GT tree is matched more than once within the same dataset."
)



# Check that every final retained segment existed originally
# Every segment kept in the final matched datasets should be traceable back to the original RCT candidate table

for name, final_df in {
    "Single sensor": easy_single_final,
    "Dual sensor": easy_dual_final,
    "Hovermap": easy_hovermap_final,
}.items():

    for dataset in final_df["dataset"].unique():

        # Original candidate segment IDs for this scanner dataset        
        original_ids = set(
            easy_candidate_qc.loc[
                easy_candidate_qc["dataset"] == dataset,
                "segment_id"
            ]
        )
        
        # Segment IDs retained in the final matched file
        final_ids = set(
            final_df.loc[
                final_df["dataset"] == dataset,
                "segment_id"
            ]
        )
        
        # Any ID remaining here would indicate that a final tree
        # cannot be traced back to the original candidate table.
        missing_ids = final_ids - original_ids

        require(
            len(missing_ids) == 0,
            f"{name}: final segment IDs not found in original candidates: "
            f"{sorted(missing_ids)}"
        )


print("All Easy terrain integrity checks passed.")

Dataset labels in Easy candidate table:
et_hovermap_nogrid      11
et_dvojskener_nogrid    10
et_j_nogrid             10
Name: dataset, dtype: int64

All Easy terrain integrity checks passed.


In [7]:
# Standardise scanner names and check the old RCT_tree_id field
# The Easy-terrain files use dataset names such as et_j_nogrid and not the scanner labels used in the dissertation
# These are converted to Single sensor, Dual sensor and Hovermap so that the same labels are used throughout the analysis
# Some older files also contain RCT_tree_id
# Before removing it,check that it is simply segment_id + 1, as expected.

EASY_DATASET_TO_SCANNER = {
    "et_j_nogrid": "Single sensor",
    "et_dvojskener_nogrid": "Dual sensor",
    "et_hovermap_nogrid": "Hovermap",
}


# Check that every dataset name in the candidate table has
# a corresponding scanner label.
unmapped_datasets = (
    set(easy_candidate_qc["dataset"].dropna().unique())
    - set(EASY_DATASET_TO_SCANNER)
)

require(
    len(unmapped_datasets) == 0,
    f"Unrecognised Easy dataset labels: {sorted(unmapped_datasets)}"
)


# Check the relationship between RCT_tree_id and segment_id
# Wherever both columns are present, RCT_tree_id should always equal segment_id + 1
# This confirms that it is only a shifted version of the same identifier rather than a separate tree ID

for name, df in {
    "GT matches": easy_gt_matches,
    "Candidate QC": easy_candidate_qc,
    "Single sensor final attributes": easy_single_final,
    "Dual sensor final attributes": easy_dual_final,
    "Hovermap final attributes": easy_hovermap_final,
}.items():

    if {"RCT_tree_id", "segment_id"}.issubset(df.columns):

        valid_ids = df[["RCT_tree_id", "segment_id"]].dropna()

        relationship_is_correct = (
            valid_ids["RCT_tree_id"]
            == valid_ids["segment_id"] + 1
        ).all()

        require(
            relationship_is_correct,
            f"{name}: RCT_tree_id is not consistently segment_id + 1."
        )


print("Legacy RCT_tree_id check passed.")


# Create cleaned working copies
# Copies are made before any labels or columns are changed so that the original loaded tables remain available unchanged


easy_gt = easy_gt_matches.copy()
easy_candidates = easy_candidate_qc.copy()

easy_single_before = easy_single_original.copy()
easy_dual_before = easy_dual_original.copy()
easy_hovermap_before = easy_hovermap_original.copy()

easy_single_after = easy_single_final.copy()
easy_dual_after = easy_dual_final.copy()
easy_hovermap_after = easy_hovermap_final.copy()


# Add consistent scanner labels
# Convert the dataset names to the scanner labels used in the dissertation for the GT-match and candidate tables
easy_gt["Scanner"] = easy_gt["dataset"].map(EASY_DATASET_TO_SCANNER)
easy_candidates["Scanner"] = easy_candidates["dataset"].map(
    EASY_DATASET_TO_SCANNER
)

# The individual scanner attribute files already represent one scanner each, so the scanner label can be added directly
easy_single_before["Scanner"] = "Single sensor"
easy_dual_before["Scanner"] = "Dual sensor"
easy_hovermap_before["Scanner"] = "Hovermap"

easy_single_after["Scanner"] = "Single sensor"
easy_dual_after["Scanner"] = "Dual sensor"
easy_hovermap_after["Scanner"] = "Hovermap"


# Remove the old RCT_tree_id field
# RCT_tree_id is only segment_id shifted by +1 and is not needed in the cleaned analysis
# segment_id is used consistently from this point onwards
for df in [
    easy_gt,
    easy_candidates,
    easy_single_after,
    easy_dual_after,
    easy_hovermap_after,
]:
    if "RCT_tree_id" in df.columns:
        df.drop(columns="RCT_tree_id", inplace=True)

# Checks of the cleaned labels and ID columns
print("\nScanner labels:")
print(easy_candidates["Scanner"].value_counts())

print("\nRCT_tree_id present in cleaned candidate table:",
      "RCT_tree_id" in easy_candidates.columns)

Legacy RCT_tree_id check passed.

Scanner labels:
Hovermap         11
Single sensor    10
Dual sensor      10
Name: Scanner, dtype: int64

RCT_tree_id present in cleaned candidate table: False


## Easy terrain: before- and after-QC measurements

The original RCT attribute files contain the measurements before manual segmentation QC. The final attribute-match files contain the measurements retained after QC and reprocessing.

These are combined using `dataset` and `segment_id`, so that each final tree can be traced back to its original RCT segment.

In [8]:
# Combine the original Easy-terrain RCT measurements
# These tables contain the tree attributes produced by RCT before any manual segmentation QC was carried out
# Only the fields needed to identify each segment and compare its original TH and DBH with the final post-QC values are kept

easy_before = pd.concat(
    [
        easy_single_before,
        easy_dual_before,
        easy_hovermap_before,
    ],
    ignore_index=True,
)

# Keep only the identifiers and tree attributes needed for the before/after QC comparison
easy_before = easy_before[
    [
        "dataset",
        "Scanner",
        "segment_id",
        "height",
        "DBH",
    ]
].copy()

# Rename the original RCT measurements so they are distinguished from the final post-QC values
easy_before = easy_before.rename(
    columns={
        "height": "TH_before",
        "DBH": "DBH_before",
    }
)


print("Original Easy RCT candidates:", len(easy_before))
print(easy_before["Scanner"].value_counts())


# Combine the final Easy-terrain attribute tables
# These files contain the final one-to-one GT-to-RCT matches retained for the attribute-accuracy analysis after QC
# Some RCT measurements remained unchanged, and others were recalculated after manual cleaning of the segmented tree

easy_after = pd.concat(
    [
        easy_single_after,
        easy_dual_after,
        easy_hovermap_after,
    ],
    ignore_index=True,
)

# Keep the GT measurements, final RCT measurements and QC information needed for the later accuracy analysis
easy_after = easy_after[
    [
        "dataset",
        "Scanner",
        "segment_id",
        "GT_ID",
        "GT_TH",
        "GT_DBH",
        "RCT_height",
        "RCT_DBH",
        "attribute_value_source",
        "QC_class",
        "include_in_attribute_accuracy",
    ]
].copy()

# Rename the final RCT measurements to make the before/after comparison easier to follow
easy_after = easy_after.rename(
    columns={
        "RCT_height": "TH_after",
        "RCT_DBH": "DBH_after",
    }
)


print("\nFinal Easy one-to-one matches:", len(easy_after))
print(easy_after["Scanner"].value_counts())


# Link each final tree to its original RCT measurement 
# dataset and segment_id are both used for the join because the same segment_id can occur in different scanner datasets
# validate="one_to_one" also checks that each final record links to only one original record, 
# and that duplicate IDs have not been introduced

easy_before_after = easy_after.merge(
    easy_before,
    on=["dataset", "Scanner", "segment_id"],
    how="left",
    validate="one_to_one",
)


# Check that the join worked correctly
# Every final retained tree should have corresponding TH and DBH values in the original RCT output

missing_before = easy_before_after[
    ["TH_before", "DBH_before"]
].isna().any(axis=1).sum()


require(
    missing_before == 0,
    "At least one final Easy tree could not be linked to its "
    "original RCT measurements."
)


# There should be nine retained trees for each of the three scanners, giving 27 scanner-tree records in total
require(
    len(easy_before_after) == 27,
    "Expected 27 Easy scanner-tree records after joining."
)


print("\nEasy before/after table created successfully.")
print("Rows:", len(easy_before_after))

print("\nRows per scanner:")
print(easy_before_after["Scanner"].value_counts())

# Display a small sample to check that the original and final
# measurements have been linked to the correct GT tree
easy_before_after[
    [
        "Scanner",
        "segment_id",
        "GT_ID",
        "GT_TH",
        "TH_before",
        "TH_after",
        "GT_DBH",
        "DBH_before",
        "DBH_after",
        "attribute_value_source",
    ]
].head(10)

Original Easy RCT candidates: 31
Hovermap         11
Single sensor    10
Dual sensor      10
Name: Scanner, dtype: int64

Final Easy one-to-one matches: 27
Single sensor    9
Dual sensor      9
Hovermap         9
Name: Scanner, dtype: int64

Easy before/after table created successfully.
Rows: 27

Rows per scanner:
Single sensor    9
Dual sensor      9
Hovermap         9
Name: Scanner, dtype: int64


,Scanner,segment_id,GT_ID,GT_TH,TH_before,TH_after,GT_DBH,DBH_before,DBH_after,attribute_value_source
0,Single sensor,0,21,31.4,29.6019,29.6801,0.627,0.5106,0.4994,cleaned
1,Single sensor,1,13,21.9,22.3470,22.3077,0.248,0.2292,0.2292,cleaned
2,Single sensor,2,20,22.5,26.3665,26.3665,0.453,0.3290,0.3290,original
3,Single sensor,3,17,16.8,19.9813,19.9813,0.351,0.2686,0.2686,original
4,Single sensor,4,27,30.5,28.3266,28.2832,0.488,0.4376,0.4348,cleaned
5,Single sensor,6,6,27.7,26.9184,26.9184,0.631,0.5086,0.5086,original
6,Single sensor,7,33,27.8,28.9578,28.9578,0.596,0.4658,0.4658,original
7,Single sensor,8,34,25.2,26.7968,27.0095,0.549,0.4474,0.4480,cleaned
8,Single sensor,9,31,23.9,25.5125,25.4234,0.558,0.4556,0.4552,cleaned
9,Dual sensor,0,21,31.4,30.4447,30.4870,0.627,0.5148,0.5026,cleaned


In [9]:
# Check how the final Easy-terrain measurements were produced
# Before calculating errors, check which final measurements were kept from the original RCT output
# Check which final measurements were recalculated after manual cleaning of the segmented trees.
# This makes it clear where the final TH and DBH values came from

print("Attribute value sources:")
print(
    easy_before_after["attribute_value_source"]
    .value_counts(dropna=False)
)


# Check that all retained rows are included in the analysis
# Every row in this table should belong to the final attribute-accuracy sample
require(
    easy_before_after["include_in_attribute_accuracy"].fillna(False).all(),
    "At least one retained Easy row is not marked for attribute accuracy."
)


# Identify measurements that changed after QC
# Compare the original and final TH and DBH values to identify trees whose measurements were updated during manual QC
# np.isclose() is used because the measurements are floating-point values and may contain very small numerical differences.

easy_before_after["TH_changed"] = ~np.isclose(
    easy_before_after["TH_before"],
    easy_before_after["TH_after"],
    equal_nan=True,
)

easy_before_after["DBH_changed"] = ~np.isclose(
    easy_before_after["DBH_before"],
    easy_before_after["DBH_after"],
    equal_nan=True,
)

# Keep only rows where at least one attribute changed so the effect of QC can be inspected directly
changed_easy = easy_before_after.loc[
    easy_before_after["TH_changed"]
    | easy_before_after["DBH_changed"],
    [
        "Scanner",
        "segment_id",
        "GT_ID",
        "TH_before",
        "TH_after",
        "DBH_before",
        "DBH_after",
        "attribute_value_source",
    ],
].copy()


print("\nRows with a changed TH and/or DBH value:")
print(f"{len(changed_easy)} of {len(easy_before_after)}")

changed_easy

Attribute value sources:
cleaned                            13
original                           12
cleaned_merged_H0_into_H1           1
original_reprocessing_exception     1
Name: attribute_value_source, dtype: int64

Rows with a changed TH and/or DBH value:
14 of 27


,Scanner,segment_id,GT_ID,TH_before,TH_after,DBH_before,DBH_after,attribute_value_source
0,Single sensor,0,21,29.6019,29.6801,0.5106,0.4994,cleaned
1,Single sensor,1,13,22.3470,22.3077,0.2292,0.2292,cleaned
4,Single sensor,4,27,28.3266,28.2832,0.4376,0.4348,cleaned
7,Single sensor,8,34,26.7968,27.0095,0.4474,0.4480,cleaned
8,Single sensor,9,31,25.5125,25.4234,0.4556,0.4552,cleaned
9,Dual sensor,0,21,30.4447,30.4870,0.5148,0.5026,cleaned
10,Dual sensor,1,13,24.0135,22.6070,0.2310,0.2296,cleaned
11,Dual sensor,2,20,26.7186,26.6743,0.3368,0.3326,cleaned
13,Dual sensor,4,27,28.8655,29.0049,0.4210,0.4190,cleaned
16,Dual sensor,8,34,27.5779,25.2756,0.4540,0.4564,cleaned


In [10]:
# Create the Easy-terrain per-tree measurement table
# Put the Easy-terrain data into the same column structure that will later be used for the Intermediate and Challenging data
# Signed errors are calculated as: estimated value - ground-truth value
# Positive errors therefore indicate overestimation, and negative errors indicate underestimation


easy_measurements = easy_before_after.copy()

# Add the terrain label so this table can later be combined
# with the other two terrain datasets
easy_measurements["Terrain"] = "Easy"

# Use the same before/after column names that will be used for all terrains in the final combined RCT dataset
easy_measurements = easy_measurements.rename(
    columns={
        "TH_before": "before_TH",
        "TH_after": "after_TH",
        "DBH_before": "before_DBH",
        "DBH_after": "after_DBH",
    }
)



# Calculate signed errors
easy_measurements["before_TH_error"] = (
    easy_measurements["before_TH"]
    - easy_measurements["GT_TH"]
)

easy_measurements["after_TH_error"] = (
    easy_measurements["after_TH"]
    - easy_measurements["GT_TH"]
)

easy_measurements["before_DBH_error"] = (
    easy_measurements["before_DBH"]
    - easy_measurements["GT_DBH"]
)

easy_measurements["after_DBH_error"] = (
    easy_measurements["after_DBH"]
    - easy_measurements["GT_DBH"]
)


# Calculate absolute errors
# Absolute error ignores the direction of the difference and shows only how far each estimate is from the ground truth
easy_measurements["before_TH_abs_error"] = (
    easy_measurements["before_TH_error"].abs()
)

easy_measurements["after_TH_abs_error"] = (
    easy_measurements["after_TH_error"].abs()
)

easy_measurements["before_DBH_abs_error"] = (
    easy_measurements["before_DBH_error"].abs()
)

easy_measurements["after_DBH_abs_error"] = (
    easy_measurements["after_DBH_error"].abs()
)


# Calculate the change in absolute error after QC
# This compares the final absolute error with the original one: after QC absolute error - before QC absolute error
# Negative values mean the error became smaller after QC
# Positive values mean the error increased

easy_measurements["TH_delta_abs_error"] = (
    easy_measurements["after_TH_abs_error"]
    - easy_measurements["before_TH_abs_error"]
)

easy_measurements["DBH_delta_abs_error"] = (
    easy_measurements["after_DBH_abs_error"]
    - easy_measurements["before_DBH_abs_error"]
)


# Check the final Easy measurement table
# These columns should all contain valid measurements for every retained Easy scanner-tree record
measurement_columns = [
    "GT_TH",
    "before_TH",
    "after_TH",
    "GT_DBH",
    "before_DBH",
    "after_DBH",
]

require(
    not easy_measurements[measurement_columns].isna().any().any(),
    "Missing TH or DBH values found in the Easy measurement table."
)

# Nine GT trees are represented for each of the three scanners,
# giving 27 scanner-tree records in total
require(
    len(easy_measurements) == 27,
    "The final Easy measurement table should contain 27 records."
)


print("Easy measurement table created successfully.")
print("Rows:", len(easy_measurements))

print("\nMean change in absolute error by scanner:")
# A negative mean change indicates that QC reduced the absolute error on average for that scanner and attribute
print(
    easy_measurements
    .groupby("Scanner")[
        ["TH_delta_abs_error", "DBH_delta_abs_error"]
    ]
    .mean()
)

# Display a small sample of the final per-tree measurements and calculated errors
easy_measurements[
    [
        "Terrain",
        "Scanner",
        "segment_id",
        "GT_ID",
        "GT_TH",
        "before_TH",
        "after_TH",
        "before_TH_error",
        "after_TH_error",
        "GT_DBH",
        "before_DBH",
        "after_DBH",
        "before_DBH_error",
        "after_DBH_error",
        "attribute_value_source",
    ]
].head()

Easy measurement table created successfully.
Rows: 27

Mean change in absolute error by scanner:
               TH_delta_abs_error  DBH_delta_abs_error
Scanner                                               
Dual sensor             -0.437200             0.001933
Hovermap                 0.028511             0.000800
Single sensor            0.005500             0.001533


,Terrain,Scanner,segment_id,GT_ID,GT_TH,before_TH,after_TH,before_TH_error,after_TH_error,GT_DBH,before_DBH,after_DBH,before_DBH_error,after_DBH_error,attribute_value_source
0,Easy,Single sensor,0,21,31.4,29.6019,29.6801,-1.7981,-1.7199,0.627,0.5106,0.4994,-0.1164,-0.1276,cleaned
1,Easy,Single sensor,1,13,21.9,22.3470,22.3077,0.4470,0.4077,0.248,0.2292,0.2292,-0.0188,-0.0188,cleaned
2,Easy,Single sensor,2,20,22.5,26.3665,26.3665,3.8665,3.8665,0.453,0.3290,0.3290,-0.1240,-0.1240,original
3,Easy,Single sensor,3,17,16.8,19.9813,19.9813,3.1813,3.1813,0.351,0.2686,0.2686,-0.0824,-0.0824,original
4,Easy,Single sensor,4,27,30.5,28.3266,28.2832,-2.1734,-2.2168,0.488,0.4376,0.4348,-0.0504,-0.0532,cleaned


## Easy terrain: validation against previous results

As a reproducibility check, the final post-QC summary statistics were calculated directly from the per-tree data and compared with the summary files produced during the original analysis.

The validation files are used only as a check. They are not used to calculate the cleaned results.

In [11]:
# Easy-terrain validation files
# These summary files were produced during the earlier version of the analysis
# They are not used as inputs here
# They are only used as a check that the cleaned workflow reproduces the same final post-QC results


EASY_VALIDATION_DIR = VALIDATION_DIR / "easy"

EASY_VALIDATION_FILES = {
    "Single sensor":
        EASY_VALIDATION_DIR / "easy_jednoskenner_attribute_summary.csv",

    "Dual sensor":
        EASY_VALIDATION_DIR / "easy_dvojskenner_attribute_summary.csv",

    "Hovermap":
        EASY_VALIDATION_DIR / "easy_hovermap_attribute_summary.csv",
}

# Check that the three Easy-terrain validation files are available
for scanner, path in EASY_VALIDATION_FILES.items():
    require(
        path.exists(),
        f"Missing Easy validation file for {scanner}: {path.name}"
    )


print("All Easy validation files found.")


# Recalculate the final post-QC summary statistics
# The final TH and DBH measurements are compared with the ground-truth values using RMSE, MAE, bias and R²
# Bias is based on signed error, so positive values indicate overestimation and negative values indicate underestimation
# For this validation, R² is calculated using the same prediction-style definition used in the original RCT summary files:
# R² = 1 - SSE / SST
# Keeping the same definition here allows the recalculated values to be compared directly with the earlier summaries

def prediction_r2(reference, estimate):
    """Calculate prediction-style R² using 1 - SSE/SST."""

    reference = np.asarray(reference, dtype=float)
    estimate = np.asarray(estimate, dtype=float)

    # Sum of squared prediction errors
    sse = np.sum((estimate - reference) ** 2)
    # Total variation in the ground-truth values
    sst = np.sum((reference - reference.mean()) ** 2)

    return 1 - (sse / sst)


def calculate_attribute_metrics(df, reference_col, estimate_col):
    """Calculate the main accuracy metrics for one tree attribute.."""

    # Signed error is always estimate - ground truth
    error = df[estimate_col] - df[reference_col]

    return {
        "n": len(df),
        "RMSE": np.sqrt(np.mean(error ** 2)),
        "MAE": np.mean(np.abs(error)),
        "bias": np.mean(error),
        "R2": prediction_r2(
            df[reference_col],
            df[estimate_col],
        ),
    }


# Calculate the final metrics for each scanner
easy_recalculated_rows = []


for scanner in SCANNERS:

    # Select the final Easy-terrain records for one scanner
    scanner_data = easy_measurements.loc[
        easy_measurements["Scanner"] == scanner
    ]
    # Calculate post-QC TH accuracy
    th = calculate_attribute_metrics(
        scanner_data,
        reference_col="GT_TH",
        estimate_col="after_TH",
    )
    # Calculate post-QC DBH accuracy
    dbh = calculate_attribute_metrics(
        scanner_data,
        reference_col="GT_DBH",
        estimate_col="after_DBH",
    )
    
    # Store the TH and DBH results in one row per scanner
    easy_recalculated_rows.append(
        {
            "Scanner": scanner,
            "TH_n": th["n"],
            "TH_RMSE": th["RMSE"],
            "TH_MAE": th["MAE"],
            "TH_bias": th["bias"],
            "TH_R2": th["R2"],
            "DBH_n": dbh["n"],
            "DBH_RMSE": dbh["RMSE"],
            "DBH_MAE": dbh["MAE"],
            "DBH_bias": dbh["bias"],
            "DBH_R2": dbh["R2"],
        }
    )

# Combine the recalculated scanner results into one summary table
easy_recalculated_summary = pd.DataFrame(
    easy_recalculated_rows
)
# Display the recalculated Easy-terrain results
easy_recalculated_summary

All Easy validation files found.


,Scanner,TH_n,TH_RMSE,TH_MAE,TH_bias,TH_R2,DBH_n,DBH_RMSE,DBH_MAE,DBH_bias,DBH_R2
0,Single sensor,9,2.125386,1.851611,0.803100,0.760682,9,0.102379,0.095822,-0.095822,0.308042
1,Dual sensor,9,2.049010,1.617100,0.982078,0.777573,9,0.102113,0.096467,-0.096467,0.311633
2,Hovermap,9,2.154547,1.726689,1.553911,0.754070,9,0.128181,0.117133,-0.117133,-0.084687


In [12]:
# Compare the recalculated results with the earlier summaries

old_easy_summaries = []


for scanner, path in EASY_VALIDATION_FILES.items():

    old = pd.read_csv(path).copy()

    # Use the standard scanner labels so the earlier summaries can be compared directly with the cleaned analysis
    old["Scanner"] = scanner

    old_easy_summaries.append(old)


# Combine the three earlier scanner summaries into one table
old_easy_summary = pd.concat(
    old_easy_summaries,
    ignore_index=True,
)


# Metrics that should match between the recalculated results and the earlier Easy-terrain summary files:
metric_columns = [
    "TH_n",
    "TH_RMSE",
    "TH_MAE",
    "TH_bias",
    "TH_R2",
    "DBH_n",
    "DBH_RMSE",
    "DBH_MAE",
    "DBH_bias",
    "DBH_R2",
]


# Join the recalculated and earlier results by scanner
comparison = easy_recalculated_summary.merge(
    old_easy_summary[
        ["Scanner"] + metric_columns
    ],
    on="Scanner",
    suffixes=("_new", "_old"),
    validate="one_to_one",
)


# Check that every recalculated metric matches the value stored in the earlier summary files within a very small tolerance
for metric in metric_columns:

    require(
        np.allclose(
            comparison[f"{metric}_new"],
            comparison[f"{metric}_old"],
            rtol=1e-8,
            atol=1e-10,
        ),
        f"Easy-terrain validation failed for {metric}."
    )


print(
    "Easy-terrain validation passed: the recalculated post-QC "
    "statistics match the earlier summary files."
)

Easy-terrain validation passed: the recalculated post-QC statistics match the earlier summary files.


# Intermediate terrain

The Intermediate-terrain RCT data is then reconstructed using the same approach as for Easy terrain.
The available source, manual-QC and validation files are listed before loading anything so that the exact files used in the reconstruction are recorded clearly.

In [13]:
# Inspect the Intermediate-terrain files
# List the files currently stored in the Intermediate input, manual QC and validation folders before defining the inputs
# This helps confirm the exact filenames used by the cleaned workflow 

INTERMEDIATE_MANUAL_QC_DIR = MANUAL_QC_DIR / "intermediate"
INTERMEDIATE_VALIDATION_DIR = VALIDATION_DIR / "intermediate"


def list_files(folder):
    """List the files stored directly within a project folder."""

    print(f"\n{folder.relative_to(PROJECT_DIR)}")

    if not folder.exists():
        print("Folder not found")
        return

    files = sorted(
        path.name
        for path in folder.iterdir()
        if path.is_file()
    )

    for filename in files:
        print(f"  {filename}")


# Check the available Intermediate-terrain source, QC and validation files before loading them
list_files(INTERMEDIATE_DIR)
list_files(INTERMEDIATE_MANUAL_QC_DIR)
list_files(INTERMEDIATE_VALIDATION_DIR)


data/raw/rct/intermediate
  intermediate_GT_to_RCT_matches.csv
  intermediate_GT_to_RCT_matches_cleanedQC.csv
  intermediate_RCT_candidate_QC.csv
  intermediate_RCT_candidate_QC_cleanedQC.csv
  intermediate_all_cleaned_attributes(in).csv

data/manual_qc/rct/intermediate
  intermediate_manual_QC_decisions.csv

data/validation/rct/intermediate
  intermediate_cleanedQC_v2_attribute_accuracy_summary.csv
  intermediate_cleanedQC_v2_detection_summary.csv
  intermediate_cleanedQC_v2_per_tree_errors.csv
  intermediate_cleanedQC_v2_validated_candidate_QC.csv
  intermediate_cleanedQC_v2_validated_primary_matches.csv


In [14]:
# Define the Intermediate-terrain input files
# The Intermediate workflow differs from Easy because both the original and cleaned GT-to-RCT match tables are available
# These provide the before- and after-QC measurements directly
# The cleaned attribute file is also kept
# It contains the recalculated measurements used to update the final Intermediate results after manual QC


INTERMEDIATE_FILES = {
    "GT matches before QC":
        INTERMEDIATE_DIR / "intermediate_GT_to_RCT_matches.csv",

    "GT matches after QC":
        INTERMEDIATE_DIR / "intermediate_GT_to_RCT_matches_cleanedQC.csv",

    "Candidate QC before cleaning":
        INTERMEDIATE_DIR / "intermediate_RCT_candidate_QC.csv",

    "Candidate QC after cleaning":
        INTERMEDIATE_DIR / "intermediate_RCT_candidate_QC_cleanedQC.csv",

    "Cleaned attributes":
        INTERMEDIATE_DIR / "intermediate_all_cleaned_attributes(in).csv",

    "Manual QC decisions":
        INTERMEDIATE_MANUAL_QC_DIR / "intermediate_manual_QC_decisions.csv",
}


# Check that all Intermediate input files are available

missing_intermediate_files = []

for name, path in INTERMEDIATE_FILES.items():

    # Report whether each expected file can be found
    if path.exists():
        print(f"OK      {name}")

    else:
        print(f"MISSING {name}")
        missing_intermediate_files.append(path)


# Stop here if any required Intermediate file is missing
if missing_intermediate_files:
    raise FileNotFoundError(
        "One or more Intermediate-terrain input files are missing. "
        "Check the folder structure before continuing."
    )

OK      GT matches before QC
OK      GT matches after QC
OK      Candidate QC before cleaning
OK      Candidate QC after cleaning
OK      Cleaned attributes
OK      Manual QC decisions


In [15]:
# Load the Intermediate terrain source files
# I load the source tables separately so that the original and cleaned-QC versions remain easy to compare
# Nothing is edited or overwritten at this stage

intermediate_matches_before = pd.read_csv(
    INTERMEDIATE_FILES["GT matches before QC"]
)

intermediate_matches_after = pd.read_csv(
    INTERMEDIATE_FILES["GT matches after QC"]
)

intermediate_candidates_before = pd.read_csv(
    INTERMEDIATE_FILES["Candidate QC before cleaning"]
)

intermediate_candidates_after = pd.read_csv(
    INTERMEDIATE_FILES["Candidate QC after cleaning"]
)

intermediate_cleaned_attributes = pd.read_csv(
    INTERMEDIATE_FILES["Cleaned attributes"]
)

intermediate_manual_qc = pd.read_csv(
    INTERMEDIATE_FILES["Manual QC decisions"]
)

# Group the loaded tables together so they can all be inspected using the same checks
intermediate_tables = {
    "GT matches before QC": intermediate_matches_before,
    "GT matches after QC": intermediate_matches_after,
    "Candidate QC before cleaning": intermediate_candidates_before,
    "Candidate QC after cleaning": intermediate_candidates_after,
    "Cleaned attributes": intermediate_cleaned_attributes,
    "Manual QC decisions": intermediate_manual_qc,
}

# Check the size and column structure of each Intermediate table before any cleaning or matching is carried out
for name, df in intermediate_tables.items():

    print("\n" + "=" * 70)
    print(name)
    print(f"Rows: {len(df)}")
    print(f"Columns: {len(df.columns)}")
    print("Column names:")
    print(df.columns.tolist())


GT matches before QC
Rows: 66
Columns: 22
Column names:
['dataset', 'GT_ID', 'GT_X', 'GT_Y', 'GT_TH', 'GT_DBH', 'RCT_tree_id', 'segment_id', 'RCT_X', 'RCT_Y', 'RCT_height', 'RCT_DBH', 'crown_radius', 'total_vol_L', 'match_distance_m', 'nearest_candidate_distance_m', 'second_nearest_candidate_distance_m', 'ambiguity_gap_m', 'assigned_is_nearest', 'review_status', 'review_reason', 'source_candidate_csv']

GT matches after QC
Rows: 66
Columns: 22
Column names:
['dataset', 'GT_ID', 'GT_X', 'GT_Y', 'GT_TH', 'GT_DBH', 'RCT_tree_id', 'segment_id', 'RCT_X', 'RCT_Y', 'RCT_height', 'RCT_DBH', 'crown_radius', 'total_vol_L', 'match_distance_m', 'nearest_candidate_distance_m', 'second_nearest_candidate_distance_m', 'ambiguity_gap_m', 'assigned_is_nearest', 'review_status', 'review_reason', 'source_candidate_csv']

Candidate QC before cleaning
Rows: 72
Columns: 15
Column names:
['dataset', 'RCT_tree_id', 'segment_id', 'RCT_X', 'RCT_Y', 'RCT_height', 'RCT_DBH', 'crown_radius', 'total_vol_L', 'candid

In [16]:
# Check the integrity of the Intermediate-terrain data
# Before comparing the measurements before and after QC:
# check that the expected records are present and that the same tree IDs are represented in both versions of the data
# The dataset and scanner labels are also inspected before they are standardised so the structure of the source files is clear


print("Dataset labels in Intermediate candidate table:")
print(intermediate_candidates_before["dataset"].value_counts())

print("\nDataset labels in cleaned attribute table:")
print(intermediate_cleaned_attributes["dataset"].value_counts())

print("\nScanner labels in cleaned attribute table:")
print(intermediate_cleaned_attributes["scanner"].value_counts())


# Check the expected number of records

require(
    len(intermediate_matches_before) == 66,
    "Expected 66 Intermediate GT-to-RCT matches before QC."
)

require(
    len(intermediate_matches_after) == 66,
    "Expected 66 Intermediate GT-to-RCT matches after QC."
)

require(
    len(intermediate_candidates_before) == 72,
    "Expected 72 Intermediate RCT candidates before cleaning."
)

require(
    len(intermediate_candidates_after) == 72,
    "Expected 72 Intermediate RCT candidates after cleaning."
)

require(
    len(intermediate_cleaned_attributes) == 14,
    "Expected 14 recalculated Intermediate tree records."
)

require(
    len(intermediate_manual_qc) == 6,
    "Expected 6 Intermediate manual QC decisions."
)



# Check that RCT segment IDs are unique
# In the match and candidate tables, each scanner dataset has its own dataset label
# So dataset + segment_id should identify each RCT segment uniquely
# The cleaned attribute table is structured differently because its dataset column only records the terrain
# The scanner name is stored separately, so scanner + segment_id is checked there


for name, df in {
    "Matches before QC": intermediate_matches_before,
    "Matches after QC": intermediate_matches_after,
    "Candidates before cleaning": intermediate_candidates_before,
    "Candidates after cleaning": intermediate_candidates_after,
}.items():

    duplicates = df.duplicated(
        subset=["dataset", "segment_id"]
    ).sum()

    require(
        duplicates == 0,
        f"Duplicate dataset + segment_id combinations found in {name}."
    )


# Check scanner + segment_id in the cleaned attribute table because the dataset column only contains the terrain name
cleaned_duplicates = intermediate_cleaned_attributes.duplicated(
    subset=["scanner", "segment_id"]
).sum()

require(
    cleaned_duplicates == 0,
    "Duplicate scanner + segment_id combinations found in cleaned attributes."
)


# Each GT tree should also appear only once within each scanner in the before- and after-QC match tables
for name, df in {
    "Matches before QC": intermediate_matches_before,
    "Matches after QC": intermediate_matches_after,
}.items():

    duplicates = df.duplicated(
        subset=["dataset", "GT_ID"]
    ).sum()

    require(
        duplicates == 0,
        f"A GT tree appears more than once within a dataset in {name}."
    )
    

# Check that the same IDs are retained before and after QC
# QC should update measurements without adding or removing matched trees
# So the identifier sets should remain unchanged between the original and cleaned versions.

before_match_ids = set(
    intermediate_matches_before[
        ["dataset", "segment_id", "GT_ID"]
    ].itertuples(index=False, name=None)
)

after_match_ids = set(
    intermediate_matches_after[
        ["dataset", "segment_id", "GT_ID"]
    ].itertuples(index=False, name=None)
)

require(
    before_match_ids == after_match_ids,
    "The matched tree IDs differ between the before- and after-QC tables."
)


before_candidate_ids = set(
    intermediate_candidates_before[
        ["dataset", "segment_id"]
    ].itertuples(index=False, name=None)
)

after_candidate_ids = set(
    intermediate_candidates_after[
        ["dataset", "segment_id"]
    ].itertuples(index=False, name=None)
)

require(
    before_candidate_ids == after_candidate_ids,
    "The candidate IDs differ between the before- and after-cleaning tables."
)


# Check the old RCT_tree_id field
# As in the Easy-terrain data, segment_id is used as the main RCT identifier
# Before ignoring RCT_tree_id, check that it is consistently equal to segment_id + 1


for name, df in {
    "Matches before QC": intermediate_matches_before,
    "Matches after QC": intermediate_matches_after,
    "Candidates before cleaning": intermediate_candidates_before,
    "Candidates after cleaning": intermediate_candidates_after,
}.items():

    valid_ids = df[["RCT_tree_id", "segment_id"]].dropna()

    relationship_is_correct = (
        valid_ids["RCT_tree_id"]
        == valid_ids["segment_id"] + 1
    ).all()

    require(
        relationship_is_correct,
        f"{name}: RCT_tree_id is not consistently segment_id + 1."
    )


print("\nAll Intermediate-terrain integrity checks passed.")

Dataset labels in Intermediate candidate table:
it_jednoskener_nogrid    24
it_dvojskener_nogrid     24
it_hovermap_nogrid       24
Name: dataset, dtype: int64

Dataset labels in cleaned attribute table:
Intermediate    14
Name: dataset, dtype: int64

Scanner labels in cleaned attribute table:
Dvojskenner     5
Jednoskenner    5
Hovermap        4
Name: scanner, dtype: int64

All Intermediate-terrain integrity checks passed.


In [17]:
# Standardise the Intermediate-terrain scanner labels
# The Intermediate files use different scanner names depending on which stage of the analysis they came from
# They are converted to one consistent set of labels before any tables are joined

INTERMEDIATE_DATASET_TO_SCANNER = {
    "it_jednoskener_nogrid": "Single sensor",
    "it_dvojskener_nogrid": "Dual sensor",
    "it_hovermap_nogrid": "Hovermap",
}


INTERMEDIATE_CLEANED_SCANNER_MAP = {
    "Jednoskenner": "Single sensor",
    "Dvojskenner": "Dual sensor",
    "Hovermap": "Hovermap",
}


# Check that all source labels have a defined scanner mapping

unmapped_datasets = (
    set(intermediate_candidates_before["dataset"].dropna().unique())
    - set(INTERMEDIATE_DATASET_TO_SCANNER)
)

require(
    len(unmapped_datasets) == 0,
    f"Unrecognised Intermediate dataset labels: {sorted(unmapped_datasets)}"
)


unmapped_cleaned_scanners = (
    set(intermediate_cleaned_attributes["scanner"].dropna().unique())
    - set(INTERMEDIATE_CLEANED_SCANNER_MAP)
)

require(
    len(unmapped_cleaned_scanners) == 0,
    "Unrecognised scanner labels in the cleaned Intermediate attributes: "
    f"{sorted(unmapped_cleaned_scanners)}"
)



# Create cleaned working copies
# Work on copies so that the source tables loaded earlier remain unchanged
# The original CSV files are not overwritten

intermediate_before = intermediate_matches_before.copy()
intermediate_after = intermediate_matches_after.copy()

intermediate_candidates_pre = intermediate_candidates_before.copy()
intermediate_candidates_post = intermediate_candidates_after.copy()

intermediate_cleaned = intermediate_cleaned_attributes.copy()
intermediate_manual = intermediate_manual_qc.copy()


# Add the standard scanner labels
# Convert the dataset names in the match and candidate tables to the scanner labels used throughout the dissertation
intermediate_before["Scanner"] = (
    intermediate_before["dataset"]
    .map(INTERMEDIATE_DATASET_TO_SCANNER)
)

intermediate_after["Scanner"] = (
    intermediate_after["dataset"]
    .map(INTERMEDIATE_DATASET_TO_SCANNER)
)

intermediate_candidates_pre["Scanner"] = (
    intermediate_candidates_pre["dataset"]
    .map(INTERMEDIATE_DATASET_TO_SCANNER)
)

intermediate_candidates_post["Scanner"] = (
    intermediate_candidates_post["dataset"]
    .map(INTERMEDIATE_DATASET_TO_SCANNER)
)

# The cleaned attribute file stores scanner names separately, so these are standardised using the second mapping
intermediate_cleaned["Scanner"] = (
    intermediate_cleaned["scanner"]
    .map(INTERMEDIATE_CLEANED_SCANNER_MAP)
)


# Remove the old RCT_tree_id field
# segment_id is used consistently as the RCT tree identifier
for df in [
    intermediate_before,
    intermediate_after,
    intermediate_candidates_pre,
    intermediate_candidates_post,
]:
    if "RCT_tree_id" in df.columns:
        df.drop(columns="RCT_tree_id", inplace=True)


# Checks of the standardised scanner labels and ID columns
print("Scanner counts in matched data:")
print(intermediate_before["Scanner"].value_counts())

print("\nScanner counts in cleaned attributes:")
print(intermediate_cleaned["Scanner"].value_counts())

print(
    "\nRCT_tree_id present in cleaned match table:",
    "RCT_tree_id" in intermediate_before.columns
)

Scanner counts in matched data:
Single sensor    22
Dual sensor      22
Hovermap         22
Name: Scanner, dtype: int64

Scanner counts in cleaned attributes:
Dual sensor      5
Single sensor    5
Hovermap         4
Name: Scanner, dtype: int64

RCT_tree_id present in cleaned match table: False


In [18]:
# Trace the Intermediate post-QC measurements back to the cleaned attribute file
# Check that every TH or DBH value changed during QC corresponds to one of the manually cleaned Intermediate trees
# The final measurements are then compared directly with the cleaned attribute file 
# To confirm that the updated values were transferred correctly

# Join the before- and after-QC tables using the same scanner,RCT segment and GT tree
intermediate_comparison = intermediate_before[
    [
        "dataset",
        "Scanner",
        "segment_id",
        "GT_ID",
        "GT_TH",
        "GT_DBH",
        "RCT_height",
        "RCT_DBH",
    ]
].merge(
    intermediate_after[
        [
            "dataset",
            "Scanner",
            "segment_id",
            "GT_ID",
            "RCT_height",
            "RCT_DBH",
        ]
    ],
    on=["dataset", "Scanner", "segment_id", "GT_ID"],
    how="inner",
    suffixes=("_before", "_after"),
    validate="one_to_one",
)


# All 66 matched scanner-tree records should be retained when the before- and after-QC tables are joined.
require(
    len(intermediate_comparison) == 66,
    "Expected 66 matched Intermediate records after joining before and after QC."
)


# Identify measurements that changed after QC
# Compare the original and final measurements to identify trees whose TH and/or DBH values were updated during cleaning
intermediate_comparison["TH_changed"] = ~np.isclose(
    intermediate_comparison["RCT_height_before"],
    intermediate_comparison["RCT_height_after"],
    equal_nan=True,
)

intermediate_comparison["DBH_changed"] = ~np.isclose(
    intermediate_comparison["RCT_DBH_before"],
    intermediate_comparison["RCT_DBH_after"],
    equal_nan=True,
)


changed_intermediate = intermediate_comparison.loc[
    intermediate_comparison["TH_changed"]
    | intermediate_comparison["DBH_changed"]
].copy()


print(
    "Matched trees with a changed TH and/or DBH:",
    len(changed_intermediate)
)


# Check that the changed trees are the manually cleaned trees
# The cleaned attribute file identifies each tree using
# Scanner + segment_id, so the same combination is used here.

changed_keys = set(
    changed_intermediate[
        ["Scanner", "segment_id"]
    ].itertuples(index=False, name=None)
)

cleaned_keys = set(
    intermediate_cleaned[
        ["Scanner", "segment_id"]
    ].itertuples(index=False, name=None)
)


# The set of changed records should match the 14 cleaned trees exactly
require(
    changed_keys == cleaned_keys,
    "The trees changed in the final match table do not exactly "
    "match the 14 trees in the cleaned attribute file."
)


print(
    "The changed records match the 14 cleaned Intermediate trees."
)


# Check the final TH and DBH values against the cleaned file
# Join the final matched table to the cleaned measurements so the updated TH and DBH values can be compared directly
cleaned_check = intermediate_after.merge(
    intermediate_cleaned[
        [
            "Scanner",
            "segment_id",
            "height",
            "DBH",
            "source_clean_file",
        ]
    ],
    on=["Scanner", "segment_id"],
    how="inner",
    validate="one_to_one",
)


# All 14 manually cleaned trees should be matched successfully
require(
    len(cleaned_check) == 14,
    "Expected to match all 14 cleaned trees to the final match table."
)


# Check that the final TH values are the values calculated from the cleaned tree segments
require(
    np.allclose(
        cleaned_check["RCT_height"],
        cleaned_check["height"],
        rtol=1e-10,
        atol=1e-12,
    ),
    "Final RCT heights do not match the cleaned attribute file."
)


# Check the same relationship for DBH
require(
    np.allclose(
        cleaned_check["RCT_DBH"],
        cleaned_check["DBH"],
        rtol=1e-10,
        atol=1e-12,
    ),
    "Final RCT DBH values do not match the cleaned attribute file."
)


print(
    "Final TH and DBH values match the cleaned attribute file."
)


# Display the cleaned trees and their source files so the updated measurements can be traced back to their origin
cleaned_check[
    [
        "Scanner",
        "segment_id",
        "GT_ID",
        "RCT_height",
        "RCT_DBH",
        "source_clean_file",
    ]
].sort_values(
    ["Scanner", "GT_ID"]
)

Matched trees with a changed TH and/or DBH: 14
The changed records match the 14 cleaned Intermediate trees.
Final TH and DBH values match the cleaned attribute file.


,Scanner,segment_id,GT_ID,RCT_height,RCT_DBH,source_clean_file
5,Dual sensor,16,14,20.4272,0.2354,INT_D_16_GT14_clean.ply
6,Dual sensor,12,15,19.8641,0.2452,INT_D_12_GT15_clean.ply
7,Dual sensor,6,16,18.7255,0.1898,INT_D_6_GT16_clean.ply
8,Dual sensor,3,19,21.6379,0.3074,INT_D_3_GT19_clean.ply
9,Dual sensor,5,29,20.7189,0.2098,INT_D_5_GT29_clean.ply
10,Hovermap,15,14,21.3310,0.2274,INT_H_15_GT14_clean.ply
11,Hovermap,11,15,19.5764,0.2374,INT_H_11_GT15_clean.ply
12,Hovermap,5,16,18.6091,0.1824,INT_H_5_GT16_clean.ply
13,Hovermap,3,19,21.9372,0.2890,INT_H_3_GT19_clean.ply
0,Single sensor,13,14,19.8651,0.2434,INT_J_13_GT14_clean.ply


In [19]:
# Create the Intermediate before/after measurement table
# The original match table provides the before-QC measurements
# The cleaned-QC table provides the final after-QC values
# The two tables are joined using dataset, Scanner, segment_id and GT_ID
# So that each tree is compared only with its own original measurement


intermediate_measurements = intermediate_before[
    [
        "dataset",
        "Scanner",
        "segment_id",
        "GT_ID",
        "GT_TH",
        "GT_DBH",
        "RCT_height",
        "RCT_DBH",
    ]
].merge(
    intermediate_after[
        [
            "dataset",
            "Scanner",
            "segment_id",
            "GT_ID",
            "RCT_height",
            "RCT_DBH",
        ]
    ],
    on=["dataset", "Scanner", "segment_id", "GT_ID"],
    how="inner",
    suffixes=("_before", "_after"),
    validate="one_to_one",
)


# All 66 matched scanner-tree records should be retained
require(
    len(intermediate_measurements) == 66,
    "Expected 66 Intermediate before/after measurement records."
)


# Rename the RCT columns so the before- and after-QC measurements are easier to distinguish
intermediate_measurements = intermediate_measurements.rename(
    columns={
        "RCT_height_before": "before_TH",
        "RCT_height_after": "after_TH",
        "RCT_DBH_before": "before_DBH",
        "RCT_DBH_after": "after_DBH",
    }
)


# Add the terrain label for the later combined dataset
intermediate_measurements["Terrain"] = "Intermediate"


# Record where each final measurement came from
# The 14 changed records were already confirmed to match the cleaned attribute file
# They are labelled as "cleaned"
# The unchanged measurements are labelled as "original"
# For cleaned trees, the source .ply filename is also retained so the recalculated measurement can be traced back to its file

intermediate_measurements = intermediate_measurements.merge(
    intermediate_cleaned[
        [
            "Scanner",
            "segment_id",
            "source_clean_file",
        ]
    ],
    on=["Scanner", "segment_id"],
    how="left",
    validate="one_to_one",
)


intermediate_measurements["attribute_value_source"] = np.where(
    intermediate_measurements["source_clean_file"].notna(),
    "cleaned",
    "original",
)


print("Attribute value sources:")
print(
    intermediate_measurements["attribute_value_source"]
    .value_counts()
)


# Calculate signed and absolute errors
# Signed error is calculated as: estimate - ground truth

intermediate_measurements["before_TH_error"] = (
    intermediate_measurements["before_TH"]
    - intermediate_measurements["GT_TH"]
)

intermediate_measurements["after_TH_error"] = (
    intermediate_measurements["after_TH"]
    - intermediate_measurements["GT_TH"]
)

intermediate_measurements["before_DBH_error"] = (
    intermediate_measurements["before_DBH"]
    - intermediate_measurements["GT_DBH"]
)

intermediate_measurements["after_DBH_error"] = (
    intermediate_measurements["after_DBH"]
    - intermediate_measurements["GT_DBH"]
)


# Absolute error shows the size of the error regardless of whether the measurement was over- or underestimated
intermediate_measurements["before_TH_abs_error"] = (
    intermediate_measurements["before_TH_error"].abs()
)

intermediate_measurements["after_TH_abs_error"] = (
    intermediate_measurements["after_TH_error"].abs()
)

intermediate_measurements["before_DBH_abs_error"] = (
    intermediate_measurements["before_DBH_error"].abs()
)

intermediate_measurements["after_DBH_abs_error"] = (
    intermediate_measurements["after_DBH_error"].abs()
)


# Calculate the change in absolute error after QC
intermediate_measurements["TH_delta_abs_error"] = (
    intermediate_measurements["after_TH_abs_error"]
    - intermediate_measurements["before_TH_abs_error"]
)

intermediate_measurements["DBH_delta_abs_error"] = (
    intermediate_measurements["after_DBH_abs_error"]
    - intermediate_measurements["before_DBH_abs_error"]
)



# Check the final Intermediate measurement table
measurement_columns = [
    "GT_TH",
    "before_TH",
    "after_TH",
    "GT_DBH",
    "before_DBH",
    "after_DBH",
]


# Every retained record should contain complete TH and DBH values
require(
    not intermediate_measurements[
        measurement_columns
    ].isna().any().any(),
    "Missing TH or DBH values found in the Intermediate table."
)


# There should be 22 matched trees for each of the three scanners, giving 66 scanner-tree records in total
require(
    len(intermediate_measurements) == 66,
    "The Intermediate measurement table should contain 66 records."
)


print("\nIntermediate measurement table created successfully.")
print("Rows:", len(intermediate_measurements))

print("\nRows per scanner:")
print(
    intermediate_measurements["Scanner"]
    .value_counts()
)

print("\nMean change in absolute error by scanner:")

# Negative mean values indicate an average reduction in absolute error after QC for that scanner and attribute
print(
    intermediate_measurements
    .groupby("Scanner")[
        ["TH_delta_abs_error", "DBH_delta_abs_error"]
    ]
    .mean()
)

Attribute value sources:
original    52
cleaned     14
Name: attribute_value_source, dtype: int64

Intermediate measurement table created successfully.
Rows: 66

Rows per scanner:
Single sensor    22
Dual sensor      22
Hovermap         22
Name: Scanner, dtype: int64

Mean change in absolute error by scanner:
               TH_delta_abs_error  DBH_delta_abs_error
Scanner                                               
Dual sensor             -0.324991            -0.006382
Hovermap                -0.180591            -0.006186
Single sensor           -0.205205            -0.004155


In [20]:
# Load the Intermediate validation files
# These files were produced during the earlier analysis and are not used to build the cleaned dataset
# They are only used to check that the reconstructed workflow reproduces the same final Intermediate results

INTERMEDIATE_VALIDATION_FILES = {
    "Validated primary matches":
        INTERMEDIATE_VALIDATION_DIR
        / "intermediate_cleanedQC_v2_validated_primary_matches.csv",

    "Attribute accuracy summary":
        INTERMEDIATE_VALIDATION_DIR
        / "intermediate_cleanedQC_v2_attribute_accuracy_summary.csv",

    "Detection summary":
        INTERMEDIATE_VALIDATION_DIR
        / "intermediate_cleanedQC_v2_detection_summary.csv",

    "Validated candidate QC":
        INTERMEDIATE_VALIDATION_DIR
        / "intermediate_cleanedQC_v2_validated_candidate_QC.csv",
}


# Check that all validation files are available

for name, path in INTERMEDIATE_VALIDATION_FILES.items():

    require(
        path.exists(),
        f"Missing Intermediate validation file: {path.name}"
    )

    print(f"OK      {name}")



# Load the validation tables

intermediate_v2_matches = pd.read_csv(
    INTERMEDIATE_VALIDATION_FILES["Validated primary matches"]
)

intermediate_v2_summary = pd.read_csv(
    INTERMEDIATE_VALIDATION_FILES["Attribute accuracy summary"]
)

intermediate_v2_detection = pd.read_csv(
    INTERMEDIATE_VALIDATION_FILES["Detection summary"]
)

intermediate_v2_candidates = pd.read_csv(
    INTERMEDIATE_VALIDATION_FILES["Validated candidate QC"]
)



# Inspect the validation tables
# Group the validation outputs together so their size and column structure can be checked using the same code

validation_tables = {
    "Validated primary matches": intermediate_v2_matches,
    "Attribute accuracy summary": intermediate_v2_summary,
    "Detection summary": intermediate_v2_detection,
    "Validated candidate QC": intermediate_v2_candidates,
}


for name, df in validation_tables.items():

    print("\n" + "=" * 70)
    print(name)
    print(f"Rows: {len(df)}")
    print(f"Columns: {len(df.columns)}")
    print("Column names:")
    print(df.columns.tolist())

OK      Validated primary matches
OK      Attribute accuracy summary
OK      Detection summary
OK      Validated candidate QC

Validated primary matches
Rows: 66
Columns: 39
Column names:
['dataset', 'GT_ID', 'GT_X', 'GT_Y', 'GT_TH', 'GT_DBH', 'RCT_tree_id', 'segment_id', 'RCT_X', 'RCT_Y', 'RCT_height', 'RCT_DBH', 'crown_radius', 'total_vol_L', 'match_distance_m', 'nearest_candidate_distance_m', 'second_nearest_candidate_distance_m', 'ambiguity_gap_m', 'assigned_is_nearest', 'review_status', 'review_reason', 'source_candidate_csv', 'manual_GT_ID_final', 'manual_QC_class', 'manual_include_in_GT_detection', 'manual_include_in_attribute_accuracy', 'manual_QC_notes', 'GT_ID_final', 'QC_class', 'include_in_GT_detection', 'include_in_attribute_accuracy', 'QC_notes', 'scanner', 'TH_error_m', 'TH_absolute_error_m', 'TH_squared_error_m2', 'DBH_error_m', 'DBH_absolute_error_m', 'DBH_squared_error_m2']

Attribute accuracy summary
Rows: 3
Columns: 12
Column names:
['dataset', 'scanner', 'TH_n', 'T

In [21]:
# Validate the Intermediate measurements tree by tree
# The v2 primary-match file comes from the earlier analysis
# It is used here only to check that the cleaned workflow reproduces the same final measurements and calculated errors

intermediate_v2 = intermediate_v2_matches.copy()


# Standardise the scanner labels in the validation file so they match the labels used in the cleaned analysis

intermediate_v2["Scanner"] = (
    intermediate_v2["scanner"]
    .map(SCANNER_NAME_MAP)
)


unmapped_v2_scanners = intermediate_v2.loc[
    intermediate_v2["Scanner"].isna(),
    "scanner"
].dropna().unique()


require(
    len(unmapped_v2_scanners) == 0,
    "Unrecognised scanner labels in the Intermediate validation file: "
    f"{sorted(unmapped_v2_scanners)}"
)


# Join the reconstructed measurements to the earlier validation table using the identifiers for the same scanner-tree match

intermediate_validation_check = intermediate_measurements.merge(
    intermediate_v2[
        [
            "dataset",
            "Scanner",
            "segment_id",
            "GT_ID",
            "RCT_height",
            "RCT_DBH",
            "TH_error_m",
            "TH_absolute_error_m",
            "TH_squared_error_m2",
            "DBH_error_m",
            "DBH_absolute_error_m",
            "DBH_squared_error_m2",
        ]
    ],
    on=["dataset", "Scanner", "segment_id", "GT_ID"],
    how="inner",
    validate="one_to_one",
)


# All 66 Intermediate scanner-tree records should match.
require(
    len(intermediate_validation_check) == 66,
    "Expected all 66 Intermediate records to match the validation file."
)


print(
    "Matched records:",
    len(intermediate_validation_check)
)


# Compare the final post-QC TH and DBH measurements with the values stored in the earlier validation file

require(
    np.allclose(
        intermediate_validation_check["after_TH"],
        intermediate_validation_check["RCT_height"],
        rtol=1e-10,
        atol=1e-12,
    ),
    "Final TH values do not match the earlier validation file."
)


require(
    np.allclose(
        intermediate_validation_check["after_DBH"],
        intermediate_validation_check["RCT_DBH"],
        rtol=1e-10,
        atol=1e-12,
    ),
    "Final DBH values do not match the earlier validation file."
)


# Compare the calculated errors as an additional check that the same error convention is being used: estimate - ground truth

require(
    np.allclose(
        intermediate_validation_check["after_TH_error"],
        intermediate_validation_check["TH_error_m"],
        rtol=1e-10,
        atol=1e-12,
    ),
    "TH signed errors do not match the earlier validation file."
)


require(
    np.allclose(
        intermediate_validation_check["after_TH_abs_error"],
        intermediate_validation_check["TH_absolute_error_m"],
        rtol=1e-10,
        atol=1e-12,
    ),
    "TH absolute errors do not match the earlier validation file."
)


require(
    np.allclose(
        intermediate_validation_check["after_TH_error"] ** 2,
        intermediate_validation_check["TH_squared_error_m2"],
        rtol=1e-10,
        atol=1e-12,
    ),
    "TH squared errors do not match the earlier validation file."
)


require(
    np.allclose(
        intermediate_validation_check["after_DBH_error"],
        intermediate_validation_check["DBH_error_m"],
        rtol=1e-10,
        atol=1e-12,
    ),
    "DBH signed errors do not match the earlier validation file."
)


require(
    np.allclose(
        intermediate_validation_check["after_DBH_abs_error"],
        intermediate_validation_check["DBH_absolute_error_m"],
        rtol=1e-10,
        atol=1e-12,
    ),
    "DBH absolute errors do not match the earlier validation file."
)


require(
    np.allclose(
        intermediate_validation_check["after_DBH_error"] ** 2,
        intermediate_validation_check["DBH_squared_error_m2"],
        rtol=1e-10,
        atol=1e-12,
    ),
    "DBH squared errors do not match the earlier validation file."
)


print(
    "Intermediate row-level validation passed: the final TH, DBH "
    "and error values match the earlier v2 output."
)

Matched records: 66
Intermediate row-level validation passed: the final TH, DBH and error values match the earlier v2 output.


In [22]:
# Validate the Intermediate summary statistics
# Recalculate the final post-QC RMSE, MAE, bias and R² from the cleaned per-tree measurements
# Then compare them with the earlier v2 summary
# The earlier summary is used only as a validation check and does not contribute to the recalculated values

intermediate_summary_rows = []


for scanner in SCANNERS:

    scanner_data = intermediate_measurements.loc[
        intermediate_measurements["Scanner"] == scanner
    ]

    # Calculate the final TH accuracy metrics for this scanner
    th = calculate_attribute_metrics(
        scanner_data,
        reference_col="GT_TH",
        estimate_col="after_TH",
    )

    # Calculate the final DBH accuracy metrics for this scanner
    dbh = calculate_attribute_metrics(
        scanner_data,
        reference_col="GT_DBH",
        estimate_col="after_DBH",
    )

    intermediate_summary_rows.append(
        {
            "Scanner": scanner,

            "TH_n": th["n"],
            "TH_RMSE": th["RMSE"],
            "TH_MAE": th["MAE"],
            "TH_bias": th["bias"],
            "TH_R2": th["R2"],

            "DBH_n": dbh["n"],
            "DBH_RMSE": dbh["RMSE"],
            "DBH_MAE": dbh["MAE"],
            "DBH_bias": dbh["bias"],
            "DBH_R2": dbh["R2"],
        }
    )


# Combine the recalculated results into one row per scanner
intermediate_recalculated_summary = pd.DataFrame(
    intermediate_summary_rows
)

intermediate_recalculated_summary


# Standardise the scanner labels in the earlier summary so they match the labels used in the cleaned workflow

intermediate_old_summary = intermediate_v2_summary.copy()

intermediate_old_summary["Scanner"] = (
    intermediate_old_summary["scanner"]
    .map(SCANNER_NAME_MAP)
)


# Check that every scanner name in the earlier summary was mapped
unmapped_summary_scanners = intermediate_old_summary.loc[
    intermediate_old_summary["Scanner"].isna(),
    "scanner"
].dropna().unique()


require(
    len(unmapped_summary_scanners) == 0,
    "Unrecognised scanner labels in the Intermediate summary: "
    f"{sorted(unmapped_summary_scanners)}"
)


# Compare the recalculated results with the earlier v2 summary

metric_columns = [
    "TH_n",
    "TH_RMSE",
    "TH_MAE",
    "TH_bias",
    "TH_R2",
    "DBH_n",
    "DBH_RMSE",
    "DBH_MAE",
    "DBH_bias",
    "DBH_R2",
]


# Join the two summaries by scanner so each metric can be compared directly
intermediate_summary_check = (
    intermediate_recalculated_summary.merge(
        intermediate_old_summary[
            ["Scanner"] + metric_columns
        ],
        on="Scanner",
        suffixes=("_new", "_old"),
        validate="one_to_one",
    )
)


# Check that every recalculated metric matches the value stored in the earlier summary within a very small tolerance
for metric in metric_columns:

    require(
        np.allclose(
            intermediate_summary_check[f"{metric}_new"],
            intermediate_summary_check[f"{metric}_old"],
            rtol=1e-8,
            atol=1e-10,
        ),
        f"Intermediate validation failed for {metric}."
    )


print(
    "Intermediate summary validation passed: the recalculated "
    "post-QC statistics match the earlier v2 summary."
)

Intermediate summary validation passed: the recalculated post-QC statistics match the earlier v2 summary.


# Challenging terrain

The Challenging-terrain RCT data is reconstructed next.
This terrain required more manual QC because several neighbouring trees were merged or difficult to separate. I therefore keep the original candidate data, final manual QC decisions and final post-QC attribute tables separate so that each decision can be traced clearly.

In [23]:
# Inspect the Challenging-terrain files
# List the available source, manual QC, validation and archived files before defining the inputs
# This is useful for the Challenging terrain because some older manual-QC files were later corrected
# They should not be used in the final workflow

CHALLENGING_MANUAL_QC_DIR = MANUAL_QC_DIR / "challenging"
CHALLENGING_VALIDATION_DIR = VALIDATION_DIR / "challenging"
CHALLENGING_ARCHIVE_DIR = DATA_DIR / "archive" / "rct" / "challenging"


# Check the files currently available in each relevant folder
list_files(CHALLENGING_DIR)
list_files(CHALLENGING_MANUAL_QC_DIR)
list_files(CHALLENGING_VALIDATION_DIR)
list_files(CHALLENGING_ARCHIVE_DIR)


data/raw/rct/challenging
  challenging_GT_to_RCT_matches.csv
  challenging_RCT_candidate_QC.csv
  challenging_dvojskenner_final_attribute_matches.csv
  challenging_hovermap_final_attribute_matches.csv
  challenging_jednoskenner_final_attribute_matches.csv
  cht_dvojskener_nogrid_tree_attributes.csv
  cht_hovermap_nogrid_tree_attributes.csv
  cht_jednoskener_nogrid_tree_attributes.csv

data/manual_qc/rct/challenging
  challenging_dvojskenner_manual_QC.csv
  challenging_hovermap_manual_QC.csv
  challenging_jednoskenner_manual_QC.csv

data/validation/rct/challenging
  challenging_dvojskenner_attribute_summary.csv
  challenging_hovermap_attribute_summary.csv
  challenging_jednoskenner_attribute_summary.csv

data/archive/rct/challenging
Folder not found


In [24]:
# Define the Challenging-terrain input files
# The original RCT outputs are used as the before-QC source
# The final attribute-match files provide the post-QC measurements
# The final manual-QC tables are also included so the candidate decisions can be reconstructed
# Older superseded QC files are not used in this workflow.

CHALLENGING_FILES = {
    "GT matches":
        CHALLENGING_DIR / "challenging_GT_to_RCT_matches.csv",

    "Candidate QC":
        CHALLENGING_DIR / "challenging_RCT_candidate_QC.csv",

    "Single sensor original attributes":
        CHALLENGING_DIR / "cht_jednoskener_nogrid_tree_attributes.csv",

    "Dual sensor original attributes":
        CHALLENGING_DIR / "cht_dvojskener_nogrid_tree_attributes.csv",

    "Hovermap original attributes":
        CHALLENGING_DIR / "cht_hovermap_nogrid_tree_attributes.csv",

    "Single sensor final attributes":
        CHALLENGING_DIR / "challenging_jednoskenner_final_attribute_matches.csv",

    "Dual sensor final attributes":
        CHALLENGING_DIR / "challenging_dvojskenner_final_attribute_matches.csv",

    "Hovermap final attributes":
        CHALLENGING_DIR / "challenging_hovermap_final_attribute_matches.csv",

    "Single sensor manual QC":
        CHALLENGING_MANUAL_QC_DIR / "challenging_jednoskenner_manual_QC.csv",

    "Dual sensor manual QC":
        CHALLENGING_MANUAL_QC_DIR / "challenging_dvojskenner_manual_QC.csv",

    "Hovermap manual QC":
        CHALLENGING_MANUAL_QC_DIR / "challenging_hovermap_manual_QC.csv",
}


# Check that all required Challenging-terrain files are available before any of them are loaded.
missing_challenging_files = []

for name, path in CHALLENGING_FILES.items():

    if path.exists():
        print(f"OK      {name}")

    else:
        print(f"MISSING {name}")
        missing_challenging_files.append(path)


# Stop here if any required source file is missing
if missing_challenging_files:
    raise FileNotFoundError(
        "One or more Challenging-terrain source files are missing. "
        "Check the folder structure before continuing."
    )

OK      GT matches
OK      Candidate QC
OK      Single sensor original attributes
OK      Dual sensor original attributes
OK      Hovermap original attributes
OK      Single sensor final attributes
OK      Dual sensor final attributes
OK      Hovermap final attributes
OK      Single sensor manual QC
OK      Dual sensor manual QC
OK      Hovermap manual QC


In [25]:
# Load the Challenging-terrain source files
# Load each source table separately 
# So the original RCT measurements, final post-QC values and manual QC decisions remain clearly separated

challenging_gt_matches = pd.read_csv(
    CHALLENGING_FILES["GT matches"]
)

challenging_candidate_qc = pd.read_csv(
    CHALLENGING_FILES["Candidate QC"]
)

challenging_single_original = pd.read_csv(
    CHALLENGING_FILES["Single sensor original attributes"]
)

challenging_dual_original = pd.read_csv(
    CHALLENGING_FILES["Dual sensor original attributes"]
)

challenging_hovermap_original = pd.read_csv(
    CHALLENGING_FILES["Hovermap original attributes"]
)

challenging_single_final = pd.read_csv(
    CHALLENGING_FILES["Single sensor final attributes"]
)

challenging_dual_final = pd.read_csv(
    CHALLENGING_FILES["Dual sensor final attributes"]
)

challenging_hovermap_final = pd.read_csv(
    CHALLENGING_FILES["Hovermap final attributes"]
)

challenging_single_manual = pd.read_csv(
    CHALLENGING_FILES["Single sensor manual QC"]
)

challenging_dual_manual = pd.read_csv(
    CHALLENGING_FILES["Dual sensor manual QC"]
)

challenging_hovermap_manual = pd.read_csv(
    CHALLENGING_FILES["Hovermap manual QC"]
)


# Group the loaded tables together so their size and column structure can be checked using the same inspection code
challenging_tables = {
    "GT matches": challenging_gt_matches,
    "Candidate QC": challenging_candidate_qc,

    "Single sensor original attributes": challenging_single_original,
    "Dual sensor original attributes": challenging_dual_original,
    "Hovermap original attributes": challenging_hovermap_original,

    "Single sensor final attributes": challenging_single_final,
    "Dual sensor final attributes": challenging_dual_final,
    "Hovermap final attributes": challenging_hovermap_final,

    "Single sensor manual QC": challenging_single_manual,
    "Dual sensor manual QC": challenging_dual_manual,
    "Hovermap manual QC": challenging_hovermap_manual,
}


# Check the number of rows and available columns in each source table before any cleaning or matching is carried out
for name, df in challenging_tables.items():

    print("\n" + "=" * 70)
    print(name)
    print(f"Rows: {len(df)}")
    print(f"Columns: {len(df.columns)}")
    print("Column names:")
    print(df.columns.tolist())


GT matches
Rows: 81
Columns: 22
Column names:
['dataset', 'GT_ID', 'GT_X', 'GT_Y', 'GT_TH', 'GT_DBH', 'RCT_tree_id', 'segment_id', 'RCT_X', 'RCT_Y', 'RCT_height', 'RCT_DBH', 'crown_radius', 'total_vol_L', 'match_distance_m', 'nearest_candidate_distance_m', 'second_nearest_candidate_distance_m', 'ambiguity_gap_m', 'assigned_is_nearest', 'review_status', 'review_reason', 'source_candidate_csv']

Candidate QC
Rows: 128
Columns: 15
Column names:
['dataset', 'RCT_tree_id', 'segment_id', 'RCT_X', 'RCT_Y', 'RCT_height', 'RCT_DBH', 'crown_radius', 'total_vol_L', 'candidate_status', 'assigned_GT_ID', 'assigned_distance_m', 'nearest_GT_ID', 'nearest_GT_distance_m', 'source_candidate_csv']

Single sensor original attributes
Rows: 43
Columns: 14
Column names:
['dataset', 'tree_id', 'segment_id', 'height', 'crown_radius', 'dimension', 'monocotal', 'DBH', 'bend', 'branch_slope', 'x', 'y', 'z', 'total_vol_L']

Dual sensor original attributes
Rows: 39
Columns: 14
Column names:
['dataset', 'tree_id', 

In [26]:
# Check the integrity of the Challenging-terrain data
# Some Challenging-terrain cases are under-segmented, meaning that one RCT segment can correspond to more than one GT tree
# For this reason, segment_id is not expected to be unique in the GT-to-RCT match table
# The candidate tables and final one-to-one attribute tables should still contain unique scanner + segment combinations

print("Dataset labels in Challenging candidate table:")
print(challenging_candidate_qc["dataset"].value_counts())


print("\nScanner labels in final attribute tables:")

for name, df in {
    "Single sensor": challenging_single_final,
    "Dual sensor": challenging_dual_final,
    "Hovermap": challenging_hovermap_final,
}.items():

    if "scanner" in df.columns:
        print(f"{name}: {df['scanner'].dropna().unique().tolist()}")


# Check the expected number of records in each source table

require(
    len(challenging_gt_matches) == 81,
    "Expected 81 Challenging GT-to-RCT match records."
)

require(
    len(challenging_candidate_qc) == 128,
    "Expected 128 Challenging RCT candidates."
)

require(
    len(challenging_single_original) == 43,
    "Expected 43 original Single-sensor candidates."
)

require(
    len(challenging_dual_original) == 39,
    "Expected 39 original Dual-sensor candidates."
)

require(
    len(challenging_hovermap_original) == 46,
    "Expected 46 original Hovermap candidates."
)

require(
    len(challenging_single_final) == 19,
    "Expected 19 final Single-sensor attribute matches."
)

require(
    len(challenging_dual_final) == 19,
    "Expected 19 final Dual-sensor attribute matches."
)

require(
    len(challenging_hovermap_final) == 19,
    "Expected 19 final Hovermap attribute matches."
)


# Check the candidate identifiers
# Each scanner segment should appear only once in the candidate table

candidate_duplicates = challenging_candidate_qc.duplicated(
    subset=["dataset", "segment_id"]
).sum()

require(
    candidate_duplicates == 0,
    "Duplicate dataset + segment_id combinations found in "
    "the Challenging candidate table."
)


# The original RCT attribute files should also contain each segment only once within each scanner dataset

for name, df in {
    "Single sensor original": challenging_single_original,
    "Dual sensor original": challenging_dual_original,
    "Hovermap original": challenging_hovermap_original,
}.items():

    require(
        df["segment_id"].is_unique,
        f"Duplicate segment_id values found in {name}."
    )


# Check the manual-QC tables
# Each table should contain one QC decision for every original RCT candidate from the corresponding scanner

require(
    len(challenging_single_manual) == 43,
    "Expected 43 Single-sensor manual-QC records."
)

require(
    len(challenging_dual_manual) == 39,
    "Expected 39 Dual-sensor manual-QC records."
)

require(
    len(challenging_hovermap_manual) == 46,
    "Expected 46 Hovermap manual-QC records."
)


for name, df in {
    "Single sensor manual QC": challenging_single_manual,
    "Dual sensor manual QC": challenging_dual_manual,
    "Hovermap manual QC": challenging_hovermap_manual,
}.items():

    require(
        df["segment_id"].is_unique,
        f"Duplicate segment_id values found in {name}."
    )


# Check the final one-to-one attribute matches
# These tables contain only the trees retained for TH and DBH accuracy
# So each segment_id and GT_ID should appear once within each scanner

for name, df in {
    "Single sensor final": challenging_single_final,
    "Dual sensor final": challenging_dual_final,
    "Hovermap final": challenging_hovermap_final,
}.items():

    require(
        df["segment_id"].is_unique,
        f"Duplicate segment_id values found in {name}."
    )

    require(
        df["GT_ID"].is_unique,
        f"Duplicate GT_ID values found in {name}."
    )


print("\nAll Challenging-terrain integrity checks passed.")

Dataset labels in Challenging candidate table:
cht_hovermap_nogrid       46
cht_jednoskener_nogrid    43
cht_dvojskener_nogrid     39
Name: dataset, dtype: int64

Scanner labels in final attribute tables:
Single sensor: ['Jednoskenner']
Dual sensor: ['Dvojskenner']
Hovermap: ['Hovermap']

All Challenging-terrain integrity checks passed.


In [27]:
# Inspect the final Challenging manual-QC decisions
# The three scanner-specific QC files were created at different stages and use slightly different column names
# Before making them consistent, inspect the final classifications and inclusiondecisions 
# so the original manual QC choices remain clear


print("SINGLE SENSOR")
print("--------------------")

print("\nQC classes:")
print(
    challenging_single_manual["QC_class"]
    .value_counts(dropna=False)
)

print("\nIncluded in GT detection:")
print(
    challenging_single_manual["include_in_GT_detection"]
    .value_counts(dropna=False)
)

print("\nIncluded in attribute accuracy:")
print(
    challenging_single_manual["include_in_attribute_accuracy"]
    .value_counts(dropna=False)
)


print("\n\nDUAL SENSOR")
print("--------------------")

print("\nFinal QC status:")
print(
    challenging_dual_manual["final_status"]
    .value_counts(dropna=False)
)

print("\nIncluded in detection:")
print(
    challenging_dual_manual["include_detection"]
    .value_counts(dropna=False)
)

print("\nIncluded in TH/DBH accuracy:")
print(
    challenging_dual_manual["include_TH_DBH"]
    .value_counts(dropna=False)
)


print("\n\nHOVERMAP")
print("--------------------")

print("\nFinal QC status:")
print(
    challenging_hovermap_manual["final_status"]
    .value_counts(dropna=False)
)

print("\nIncluded in detection:")
print(
    challenging_hovermap_manual["include_detection"]
    .value_counts(dropna=False)
)

print("\nIncluded in TH/DBH accuracy:")
print(
    challenging_hovermap_manual["include_TH_DBH"]
    .value_counts(dropna=False)
)


# Inspect selected difficult and under-segmented cases
# The manual-QC files store GT IDs in different formats
# Some fields contain single numeric IDs
# Others contain text such as 'GT23/GT24'
# The relevant fields are therefore searched as text
# This is only used to find and display the selected records
# None of the source values are changed

def rows_containing_gt_ids(df, gt_ids, columns):
    """Find rows containing any of the selected GT IDs."""

    mask = pd.Series(False, index=df.index)

    for column in columns:

        if column not in df.columns:
            continue

        values = df[column].astype(str)

        for gt_id in gt_ids:

            # Match the complete GT number so, for example,
            # GT2 does not also match GT20.
            mask |= values.str.contains(
                rf"(?<!\d){gt_id}(?!\d)",
                regex=True,
                na=False,
            )

    return df.loc[mask]


# GT trees selected for closer inspection because they include some of the more difficult Challenging-terrain QC cases
challenging_qc_ids = [6, 7, 23, 24, 32, 35, 36, 37]


print("SINGLE SENSOR - selected QC cases")

single_selected = rows_containing_gt_ids(
    challenging_single_manual,
    challenging_qc_ids,
    [
        "GT_ID",
        "GT_ID_final",
        "represented_GT_IDs",
        "QC_notes",
    ],
)

display(
    single_selected[
        [
            "segment_id",
            "GT_ID",
            "GT_ID_final",
            "represented_GT_IDs",
            "QC_class",
            "include_in_GT_detection",
            "GT_detection_credit",
            "include_in_attribute_accuracy",
            "QC_notes",
        ]
    ]
)


print("\nDUAL SENSOR - selected QC cases")

dual_selected = rows_containing_gt_ids(
    challenging_dual_manual,
    challenging_qc_ids,
    [
        "GT_ID",
        "notes",
    ],
)

display(
    dual_selected[
        [
            "segment_id",
            "GT_ID",
            "final_status",
            "include_detection",
            "include_TH_DBH",
            "final_tree_id",
            "notes",
        ]
    ]
)


print("\nHOVERMAP - selected QC cases")

hovermap_selected = rows_containing_gt_ids(
    challenging_hovermap_manual,
    challenging_qc_ids,
    [
        "GT_ID",
        "notes",
    ],
)

display(
    hovermap_selected[
        [
            "segment_id",
            "GT_ID",
            "final_status",
            "include_detection",
            "include_TH_DBH",
            "final_tree_id",
            "notes",
        ]
    ]
)

SINGLE SENSOR
--------------------

QC classes:
validated_primary_match                       18
rejected_non_tree_segment                     10
genuine_unrecorded_tree                        8
merged_GT_trees                                4
split_fragment                                 1
validated_primary_match_after_manual_split     1
genuine_unrecorded_fallen_tree                 1
Name: QC_class, dtype: int64

Included in GT detection:
True     23
False    20
Name: include_in_GT_detection, dtype: int64

Included in attribute accuracy:
False    24
True     19
Name: include_in_attribute_accuracy, dtype: int64


DUAL SENSOR
--------------------

Final QC status:
genuine_unrecorded_tree           11
cleaned_retained_tree             11
validated_untouched_tree           6
random_or_unresolved_fragment      5
ambiguous_undersegmented           2
mixed_undersegmented               2
validated_primary_match            1
cleaned_reprocessing_exception     1
Name: final_status, dtype: in

,segment_id,GT_ID,GT_ID_final,represented_GT_IDs,QC_class,include_in_GT_detection,GT_detection_credit,include_in_attribute_accuracy,QC_notes
7,7,NaN,NaN,7;32,merged_GT_trees,True,1,False,One RCT segment merges the two separate physic...
11,11,6.0,6.0,6,validated_primary_match,True,1,True,Validated one-to-one correspondence with GT6; ...
14,14,NaN,NaN,35;36,merged_GT_trees,True,1,False,One RCT segment merges the two separate physic...
23,23,NaN,NaN,23;24,merged_GT_trees,True,1,False,One RCT segment merges the two separate physic...
25,25,37.0,37.0,37,validated_primary_match,True,1,True,Validated one-to-one correspondence with GT37;...



DUAL SENSOR - selected QC cases


,segment_id,GT_ID,final_status,include_detection,include_TH_DBH,final_tree_id,notes
6,6,GT32/GT7,ambiguous_undersegmented,False,False,6.0,D6 represents the interlocked GT32/GT7 locatio...
7,7,GT6,cleaned_retained_tree,True,True,7.0,Manual cleaning completed and cleaned cloud su...
13,13,GT35/GT36,mixed_undersegmented,True,False,13.0,D13 represents the GT35/GT36 interlocked locat...
22,22,GT23/GT24,mixed_undersegmented,True,False,22.0,One RCT instance represents the very close GT2...
23,23,GT37,validated_primary_match,True,True,23.0,Validated one-to-one correspondence with GT37 ...



HOVERMAP - selected QC cases


,segment_id,GT_ID,final_status,include_detection,include_TH_DBH,final_tree_id,notes
4,4,GT32/GT7,ambiguous_undersegmented,False,False,4.0,Manual CloudCompare QC: poor segmentation of t...
10,10,6,random_or_unresolved_fragment,True,True,NaN,Not a valid independent tree after manual QC
12,12,GT35+GT36,mixed_undersegmented,True,False,12.0,One RCT instance contains two physical GT tree...
13,13,NaN,genuine_unrecorded_tree,True,False,13.0,Genuine unrecorded tree; T37-T40 absorbed into...
21,21,GT23/GT24,mixed_undersegmented,True,False,21.0,One RCT instance represents the very close GT2...
22,22,GT37,validated_primary_match,True,True,22.0,Validated one-to-one correspondence with GT37 ...
32,32,NaN,genuine_unrecorded_tree,True,False,32.0,Manual CloudCompare QC confirmed H32 is a genu...
43,43,GT23,segmentation_unreliable,True,False,43.0,"T43 is interpreted as GT23, adjacent to T21/GT..."


In [28]:
# Inspect Challenging cases that need closer interpretation
# Some earlier QC flags were created before the final detection rules were settled
# I therefore review all merged, under-segmented and uncertain cases before creating the final cleaned QC table.
# This step is only for inspection, no source values are changed


print("SINGLE SENSOR - merged / non-standard cases")

# Select Single-sensor records with merged, split or manually corrected segmentation outcomes
single_problem_cases = challenging_single_manual.loc[
    challenging_single_manual["QC_class"].isin(
        [
            "merged_GT_trees",
            "split_fragment",
            "validated_primary_match_after_manual_split",
        ]
    )
].copy()

display(
    single_problem_cases[
        [
            "segment_id",
            "GT_ID",
            "GT_ID_final",
            "represented_GT_IDs",
            "QC_class",
            "include_in_GT_detection",
            "GT_detection_credit",
            "include_in_attribute_accuracy",
            "QC_notes",
        ]
    ]
)


print("\nDUAL SENSOR - under-segmented / uncertain cases")

# Select Dual-sensor records where the segment contains multiple trees or the final interpretation remained uncertain
dual_problem_cases = challenging_dual_manual.loc[
    challenging_dual_manual["final_status"].isin(
        [
            "mixed_undersegmented",
            "ambiguous_undersegmented",
            "random_or_unresolved_fragment",
        ]
    )
].copy()

display(
    dual_problem_cases[
        [
            "segment_id",
            "GT_ID",
            "final_status",
            "include_detection",
            "include_TH_DBH",
            "final_tree_id",
            "notes",
        ]
    ]
)


print("\nHOVERMAP - under-segmented / uncertain cases")

# Select Hovermap records with under-segmentation, unreliable segmentation or unresolved neighbouring-tree assignments
hovermap_problem_cases = challenging_hovermap_manual.loc[
    challenging_hovermap_manual["final_status"].isin(
        [
            "mixed_undersegmented",
            "ambiguous_undersegmented",
            "segmentation_unreliable",
            "ambiguous_close_neighbours",
            "random_or_unresolved_fragment",
        ]
    )
].copy()

display(
    hovermap_problem_cases[
        [
            "segment_id",
            "GT_ID",
            "final_status",
            "include_detection",
            "include_TH_DBH",
            "final_tree_id",
            "notes",
        ]
    ]
)

SINGLE SENSOR - merged / non-standard cases


,segment_id,GT_ID,GT_ID_final,represented_GT_IDs,QC_class,include_in_GT_detection,GT_detection_credit,include_in_attribute_accuracy,QC_notes
2,2,NaN,NaN,33;34,split_fragment,False,0,False,Detached secondary fragment from the GT33/GT34...
3,3,NaN,NaN,33;34,merged_GT_trees,True,1,False,One RCT segment merges the two separate physic...
7,7,NaN,NaN,7;32,merged_GT_trees,True,1,False,One RCT segment merges the two separate physic...
14,14,NaN,NaN,35;36,merged_GT_trees,True,1,False,One RCT segment merges the two separate physic...
23,23,NaN,NaN,23;24,merged_GT_trees,True,1,False,One RCT segment merges the two separate physic...
29,29,20.0,20.0,20,validated_primary_match_after_manual_split,True,1,True,Original J29 contained GT20 plus an unrecorded...



DUAL SENSOR - under-segmented / uncertain cases


,segment_id,GT_ID,final_status,include_detection,include_TH_DBH,final_tree_id,notes
2,2,NaN,random_or_unresolved_fragment,False,False,NaN,Manual CloudCompare QC: not a valid independen...
3,3,GT33/GT34,ambiguous_undersegmented,False,False,3.0,D3 represents the interlocked GT33/GT34 locati...
6,6,GT32/GT7,ambiguous_undersegmented,False,False,6.0,D6 represents the interlocked GT32/GT7 locatio...
13,13,GT35/GT36,mixed_undersegmented,True,False,13.0,D13 represents the GT35/GT36 interlocked locat...
18,18,NaN,random_or_unresolved_fragment,False,False,NaN,Manual cross-scanner QC confirmed this is a no...
22,22,GT23/GT24,mixed_undersegmented,True,False,22.0,One RCT instance represents the very close GT2...
32,32,NaN,random_or_unresolved_fragment,False,False,NaN,Manual CloudCompare QC: not a valid independen...
36,36,NaN,random_or_unresolved_fragment,False,False,NaN,Manual cross-scanner QC confirmed this is a no...
37,37,NaN,random_or_unresolved_fragment,False,False,NaN,Manual cross-scanner QC confirmed this is a no...



HOVERMAP - under-segmented / uncertain cases


,segment_id,GT_ID,final_status,include_detection,include_TH_DBH,final_tree_id,notes
2,2,GT33/GT34,ambiguous_undersegmented,False,False,2.0,Manual CloudCompare QC: neighbouring trees are...
4,4,GT32/GT7,ambiguous_undersegmented,False,False,4.0,Manual CloudCompare QC: poor segmentation of t...
10,10,6,random_or_unresolved_fragment,True,True,NaN,Not a valid independent tree after manual QC
12,12,GT35+GT36,mixed_undersegmented,True,False,12.0,One RCT instance contains two physical GT tree...
19,19,NaN,random_or_unresolved_fragment,False,False,NaN,Not a valid independent tree after manual QC
21,21,GT23/GT24,mixed_undersegmented,True,False,21.0,One RCT instance represents the very close GT2...
30,30,NaN,random_or_unresolved_fragment,False,False,NaN,Not a valid independent tree after manual QC
33,33,NaN,random_or_unresolved_fragment,False,False,NaN,Not a valid independent tree after manual QC
41,41,5,ambiguous_close_neighbours,True,True,41.0,Manual CloudCompare QC: T41 is a small tree lo...
42,42,NaN,random_or_unresolved_fragment,False,False,NaN,Not a valid independent tree after manual QC


## Final Challenging-terrain QC rules

Some of the earlier Challenging-terrain QC files were created before the final definition of tree detection was fixed.

For the final analysis, a GT tree is counted as detected only if it is recovered as a distinct individual RCT tree. An under-segmented RCT segment representing more than one GT tree therefore receives no individual-tree detection credit and is excluded from the attribute-accuracy analysis.

The original QC files are left unchanged. Any final corrections are applied below to working copies so that the decisions remain clear and reproducible.

In [29]:
# Create working copies of the final Challenging manual-QC tables
# The original loaded QC tables are kept unchanged
# The final dissertation rules are applied only to these working copies

challenging_single_qc = challenging_single_manual.copy()
challenging_dual_qc = challenging_dual_manual.copy()
challenging_hovermap_qc = challenging_hovermap_manual.copy()


# Apply the final rule for merged and under-segmented trees
# If one RCT segment represents two GT trees, neither tree is considered to have been recovered as a separate individual
# These segments therefore receive no individual-tree detection credit and are excluded from TH/DBH accuracy
# The segment itself remains in the candidate table so the segmentation problem is still recorded


# Single-sensor merged segments
single_merged_segments = [3, 7, 14, 23]

challenging_single_qc.loc[
    challenging_single_qc["segment_id"].isin(single_merged_segments),
    "include_in_GT_detection"
] = False

challenging_single_qc.loc[
    challenging_single_qc["segment_id"].isin(single_merged_segments),
    "GT_detection_credit"
] = 0

challenging_single_qc.loc[
    challenging_single_qc["segment_id"].isin(single_merged_segments),
    "include_in_attribute_accuracy"
] = False


# Dual-sensor under-segmented segments
dual_undersegmented_segments = [3, 6, 13, 22]

challenging_dual_qc.loc[
    challenging_dual_qc["segment_id"].isin(dual_undersegmented_segments),
    "include_detection"
] = False

challenging_dual_qc.loc[
    challenging_dual_qc["segment_id"].isin(dual_undersegmented_segments),
    "include_TH_DBH"
] = False


# Hovermap under-segmented segments
hovermap_undersegmented_segments = [2, 4, 12, 21]

challenging_hovermap_qc.loc[
    challenging_hovermap_qc["segment_id"].isin(
        hovermap_undersegmented_segments
    ),
    "include_detection"
] = False

challenging_hovermap_qc.loc[
    challenging_hovermap_qc["segment_id"].isin(
        hovermap_undersegmented_segments
    ),
    "include_TH_DBH"
] = False


# Apply the final Hovermap corrections
# Hovermap segment 10 was manually confirmed as GT6 and should be retained for both detection and attribute accuracy
# An older status still labels this segment as unresolved
# so the working copy is updated here to match the final manual decision

h10_mask = challenging_hovermap_qc["segment_id"] == 10

require(
    h10_mask.sum() == 1,
    "Expected exactly one Hovermap segment 10 record."
)

challenging_hovermap_qc.loc[h10_mask, "GT_ID"] = "GT6"
challenging_hovermap_qc.loc[h10_mask, "final_status"] = (
    "validated_primary_match"
)
challenging_hovermap_qc.loc[h10_mask, "include_detection"] = True
challenging_hovermap_qc.loc[h10_mask, "include_TH_DBH"] = True
challenging_hovermap_qc.loc[h10_mask, "final_tree_id"] = 10


# Hovermap segment 43 occurs in the GT23 area but does not represent a reliable separate GT23 tree
# GT23 and GT24 remain an under-segmented pair and are excluded as individual detections

h43_mask = challenging_hovermap_qc["segment_id"] == 43

require(
    h43_mask.sum() == 1,
    "Expected exactly one Hovermap segment 43 record."
)

challenging_hovermap_qc.loc[h43_mask, "include_detection"] = False
challenging_hovermap_qc.loc[h43_mask, "include_TH_DBH"] = False


# Check that the main merged and under-segmented cases now follow the final detection and attribute-accuracy rules

require(
    not challenging_single_qc.loc[
        challenging_single_qc["segment_id"].isin(single_merged_segments),
        "include_in_GT_detection"
    ].any(),
    "A merged Single-sensor candidate still has detection credit."
)

require(
    not challenging_dual_qc.loc[
        challenging_dual_qc["segment_id"].isin(dual_undersegmented_segments),
        "include_detection"
    ].any(),
    "An under-segmented Dual-sensor candidate still has detection credit."
)

require(
    not challenging_hovermap_qc.loc[
        challenging_hovermap_qc["segment_id"].isin(
            hovermap_undersegmented_segments
        ),
        "include_detection"
    ].any(),
    "An under-segmented Hovermap candidate still has detection credit."
)

require(
    challenging_hovermap_qc.loc[h10_mask, "include_detection"].iloc[0],
    "Hovermap segment 10 / GT6 should be retained for detection."
)

require(
    challenging_hovermap_qc.loc[h10_mask, "include_TH_DBH"].iloc[0],
    "Hovermap segment 10 / GT6 should be retained for attribute accuracy."
)


print("Final Challenging-terrain QC rules applied successfully.")

Final Challenging-terrain QC rules applied successfully.


In [30]:
# Show the Challenging QC records affected by the final rules
# Display the relevant rows so the final treatment of the merged, under-segmented and corrected cases is easy to inspect

print("Single sensor:")
display(
    challenging_single_qc.loc[
        challenging_single_qc["segment_id"].isin(
            single_merged_segments
        ),
        [
            "segment_id",
            "represented_GT_IDs",
            "QC_class",
            "include_in_GT_detection",
            "GT_detection_credit",
            "include_in_attribute_accuracy",
        ]
    ]
)


print("\nDual sensor:")
display(
    challenging_dual_qc.loc[
        challenging_dual_qc["segment_id"].isin(
            dual_undersegmented_segments
        ),
        [
            "segment_id",
            "GT_ID",
            "final_status",
            "include_detection",
            "include_TH_DBH",
        ]
    ]
)


print("\nHovermap:")
display(
    challenging_hovermap_qc.loc[
        challenging_hovermap_qc["segment_id"].isin(
            hovermap_undersegmented_segments + [10, 43]
        ),
        [
            "segment_id",
            "GT_ID",
            "final_status",
            "include_detection",
            "include_TH_DBH",
        ]
    ]
)

Single sensor:


,segment_id,represented_GT_IDs,QC_class,include_in_GT_detection,GT_detection_credit,include_in_attribute_accuracy
3,3,33;34,merged_GT_trees,False,0,False
7,7,7;32,merged_GT_trees,False,0,False
14,14,35;36,merged_GT_trees,False,0,False
23,23,23;24,merged_GT_trees,False,0,False



Dual sensor:


,segment_id,GT_ID,final_status,include_detection,include_TH_DBH
3,3,GT33/GT34,ambiguous_undersegmented,False,False
6,6,GT32/GT7,ambiguous_undersegmented,False,False
13,13,GT35/GT36,mixed_undersegmented,False,False
22,22,GT23/GT24,mixed_undersegmented,False,False



Hovermap:


,segment_id,GT_ID,final_status,include_detection,include_TH_DBH
2,2,GT33/GT34,ambiguous_undersegmented,False,False
4,4,GT32/GT7,ambiguous_undersegmented,False,False
10,10,GT6,validated_primary_match,True,True
12,12,GT35+GT36,mixed_undersegmented,False,False
21,21,GT23/GT24,mixed_undersegmented,False,False
43,43,GT23,segmentation_unreliable,False,False


In [31]:
# Prepare the Challenging measurement tables
# The original RCT attribute files contain the before-QC measurements for all candidates
# The final attribute files contain the one-to-one GT matches retained after QC
# Scanner labels are standardised here, but the original source files remain unchanged

challenging_single_before = challenging_single_original.copy()
challenging_dual_before = challenging_dual_original.copy()
challenging_hovermap_before = challenging_hovermap_original.copy()

challenging_single_after = challenging_single_final.copy()
challenging_dual_after = challenging_dual_final.copy()
challenging_hovermap_after = challenging_hovermap_final.copy()


# Add the standard scanner labels used throughout the analysis

challenging_single_before["Scanner"] = "Single sensor"
challenging_dual_before["Scanner"] = "Dual sensor"
challenging_hovermap_before["Scanner"] = "Hovermap"

challenging_single_after["Scanner"] = "Single sensor"
challenging_dual_after["Scanner"] = "Dual sensor"
challenging_hovermap_after["Scanner"] = "Hovermap"


# Use GT_numeric as the consistent GT identifier in the final measurement tables
# Older files store GT IDs in different formats, so this avoids mixing numeric and text-based IDs

for name, df in {
    "Single sensor final": challenging_single_after,
    "Dual sensor final": challenging_dual_after,
    "Hovermap final": challenging_hovermap_after,
}.items():

    require(
        "GT_numeric" in df.columns,
        f"{name} does not contain GT_numeric."
    )

    require(
        df["GT_numeric"].notna().all(),
        f"{name} contains missing GT_numeric values."
    )


# Combine the original Challenging RCT measurements from all three scanners into one before-QC table

challenging_before = pd.concat(
    [
        challenging_single_before,
        challenging_dual_before,
        challenging_hovermap_before,
    ],
    ignore_index=True,
)


# Keep only the identifiers and attributes needed for the before/after comparison
challenging_before = challenging_before[
    [
        "Scanner",
        "segment_id",
        "height",
        "DBH",
    ]
].copy()


challenging_before = challenging_before.rename(
    columns={
        "height": "before_TH",
        "DBH": "before_DBH",
    }
)


require(
    len(challenging_before) == 128,
    "Expected 128 original Challenging RCT candidates."
)


# Combine the final one-to-one Challenging matches retained for the attribute-accuracy analysis

challenging_after = pd.concat(
    [
        challenging_single_after,
        challenging_dual_after,
        challenging_hovermap_after,
    ],
    ignore_index=True,
)


challenging_after = challenging_after[
    [
        "Scanner",
        "segment_id",
        "GT_numeric",
        "GT_TH",
        "GT_DBH",
        "RCT_height",
        "RCT_DBH",
        "attribute_value_source",
    ]
].copy()


# Rename the GT and RCT columns so they match the structure used for the Easy and Intermediate measurement tables
challenging_after = challenging_after.rename(
    columns={
        "GT_numeric": "GT_ID",
        "RCT_height": "after_TH",
        "RCT_DBH": "after_DBH",
    }
)


# There should be 19 retained trees for each of the three scanners, giving 57 scanner-tree records in total
require(
    len(challenging_after) == 57,
    "Expected 57 final Challenging measurement records."
)


print("Final Challenging records per scanner:")
print(challenging_after["Scanner"].value_counts())

Final Challenging records per scanner:
Single sensor    19
Dual sensor      19
Hovermap         19
Name: Scanner, dtype: int64


In [32]:
# Join the Challenging before- and after-QC measurements
# Link each retained final tree back to its original RCT segment using Scanner + segment_id
# This allows the original and final measurements for the same RCT segment to be compared directly

challenging_measurements = challenging_after.merge(
    challenging_before,
    on=["Scanner", "segment_id"],
    how="left",
    validate="one_to_one",
)


# All 57 final scanner-tree records should be retained after the join
require(
    len(challenging_measurements) == 57,
    "Expected 57 Challenging before/after records."
)


# Every retained tree should be traceable back to its original RCT TH and DBH measurements
require(
    not challenging_measurements[
        ["before_TH", "before_DBH"]
    ].isna().any().any(),
    "At least one final Challenging tree could not be traced "
    "back to its original RCT attributes."
)


# Add the terrain label for the later combined dataset
challenging_measurements["Terrain"] = "Challenging"


print("Challenging before/after table created successfully.")
print("Rows:", len(challenging_measurements))

print("\nAttribute value sources:")
print(
    challenging_measurements["attribute_value_source"]
    .value_counts(dropna=False)
)


# Calculate signed and absolute errors
# Signed error is calculated as: estimate - ground truth
# Positive values indicate overestimation and negative values indicate underestimation

challenging_measurements["before_TH_error"] = (
    challenging_measurements["before_TH"]
    - challenging_measurements["GT_TH"]
)

challenging_measurements["after_TH_error"] = (
    challenging_measurements["after_TH"]
    - challenging_measurements["GT_TH"]
)

challenging_measurements["before_DBH_error"] = (
    challenging_measurements["before_DBH"]
    - challenging_measurements["GT_DBH"]
)

challenging_measurements["after_DBH_error"] = (
    challenging_measurements["after_DBH"]
    - challenging_measurements["GT_DBH"]
)


# Absolute error shows the size of the difference from the ground truth regardless of whether it was over- or underestimated
challenging_measurements["before_TH_abs_error"] = (
    challenging_measurements["before_TH_error"].abs()
)

challenging_measurements["after_TH_abs_error"] = (
    challenging_measurements["after_TH_error"].abs()
)

challenging_measurements["before_DBH_abs_error"] = (
    challenging_measurements["before_DBH_error"].abs()
)

challenging_measurements["after_DBH_abs_error"] = (
    challenging_measurements["after_DBH_error"].abs()
)


# Calculate the change in absolute error after QC
challenging_measurements["TH_delta_abs_error"] = (
    challenging_measurements["after_TH_abs_error"]
    - challenging_measurements["before_TH_abs_error"]
)

challenging_measurements["DBH_delta_abs_error"] = (
    challenging_measurements["after_DBH_abs_error"]
    - challenging_measurements["before_DBH_abs_error"]
)


# Check that all final measurement fields are complete
measurement_columns = [
    "GT_TH",
    "before_TH",
    "after_TH",
    "GT_DBH",
    "before_DBH",
    "after_DBH",
]


require(
    not challenging_measurements[
        measurement_columns
    ].isna().any().any(),
    "Missing TH or DBH values found in the Challenging table."
)


print("\nMean change in absolute error by scanner:")
print(
    challenging_measurements
    .groupby("Scanner")[
        ["TH_delta_abs_error", "DBH_delta_abs_error"]
    ]
    .mean()
)

Challenging before/after table created successfully.
Rows: 57

Attribute value sources:
cleaned                            35
original                           18
original_reprocessing_exception     4
Name: attribute_value_source, dtype: int64

Mean change in absolute error by scanner:
               TH_delta_abs_error  DBH_delta_abs_error
Scanner                                               
Dual sensor             -0.365237            -0.002411
Hovermap                -0.473989            -0.000542
Single sensor           -0.252189             0.003489


In [33]:
# Load the Challenging validation summaries
# These summary files were produced during the earlier analysis
# They are used here only to check that the cleaned workflow reproduces the same final post-QC results
# They are not used to build or calculate the new measurements

CHALLENGING_VALIDATION_FILES = {
    "Single sensor":
        CHALLENGING_VALIDATION_DIR
        / "challenging_jednoskenner_attribute_summary.csv",

    "Dual sensor":
        CHALLENGING_VALIDATION_DIR
        / "challenging_dvojskenner_attribute_summary.csv",

    "Hovermap":
        CHALLENGING_VALIDATION_DIR
        / "challenging_hovermap_attribute_summary.csv",
}


challenging_validation_summaries = {}


for scanner, path in CHALLENGING_VALIDATION_FILES.items():

    # Check that the expected validation file is available
    require(
        path.exists(),
        f"Missing Challenging validation file for {scanner}: {path.name}"
    )

    df = pd.read_csv(path)

    # Store each summary by scanner so it can be compared with the recalculated results later
    challenging_validation_summaries[scanner] = df

    # Inspect the structure and contents of each validation table
    print("\n" + "=" * 70)
    print(scanner)
    print(f"Rows: {len(df)}")
    print(f"Columns: {len(df.columns)}")
    print("Column names:")
    print(df.columns.tolist())

    print("\nContents:")
    display(df)


Single sensor
Rows: 1
Columns: 11
Column names:
['scanner', 'TH_n', 'TH_RMSE', 'TH_MAE', 'TH_bias', 'TH_R2', 'DBH_n', 'DBH_RMSE', 'DBH_MAE', 'DBH_bias', 'DBH_R2']

Contents:


,scanner,TH_n,TH_RMSE,TH_MAE,TH_bias,TH_R2,DBH_n,DBH_RMSE,DBH_MAE,DBH_bias,DBH_R2
0,Jednoskenner,19,1.708191,1.256426,-0.655689,0.899692,19,0.06442,0.0525,-0.011005,0.612721



Dual sensor
Rows: 1
Columns: 11
Column names:
['scanner', 'TH_n', 'TH_RMSE', 'TH_MAE', 'TH_bias', 'TH_R2', 'DBH_n', 'DBH_RMSE', 'DBH_MAE', 'DBH_bias', 'DBH_R2']

Contents:


,scanner,TH_n,TH_RMSE,TH_MAE,TH_bias,TH_R2,DBH_n,DBH_RMSE,DBH_MAE,DBH_bias,DBH_R2
0,Dvojskenner,19,2.270874,1.596668,-0.859416,0.822724,19,0.059845,0.045484,-0.018747,0.665772



Hovermap
Rows: 1
Columns: 11
Column names:
['scanner', 'TH_n', 'TH_RMSE', 'TH_MAE', 'TH_bias', 'TH_R2', 'DBH_n', 'DBH_RMSE', 'DBH_MAE', 'DBH_bias', 'DBH_R2']

Contents:


,scanner,TH_n,TH_RMSE,TH_MAE,TH_bias,TH_R2,DBH_n,DBH_RMSE,DBH_MAE,DBH_bias,DBH_R2
0,Hovermap,19,1.746137,1.352605,-0.397647,0.895186,19,0.073196,0.056221,-0.043705,0.500017


In [34]:
# Validate the Challenging summary statistics
# Recalculate the final post-QC RMSE, MAE, bias and R² directly from the 57 retained scanner-tree records
# Then compare them with the earlier summary files 
# The earlier summaries are used only for validation and do not contribute to the recalculated results

challenging_summary_rows = []


for scanner in SCANNERS:

    scanner_data = challenging_measurements.loc[
        challenging_measurements["Scanner"] == scanner
    ]

    # Calculate the final TH accuracy metrics for this scanner
    th = calculate_attribute_metrics(
        scanner_data,
        reference_col="GT_TH",
        estimate_col="after_TH",
    )

    # Calculate the final DBH accuracy metrics for this scanner
    dbh = calculate_attribute_metrics(
        scanner_data,
        reference_col="GT_DBH",
        estimate_col="after_DBH",
    )

    challenging_summary_rows.append(
        {
            "Scanner": scanner,

            "TH_n": th["n"],
            "TH_RMSE": th["RMSE"],
            "TH_MAE": th["MAE"],
            "TH_bias": th["bias"],
            "TH_R2": th["R2"],

            "DBH_n": dbh["n"],
            "DBH_RMSE": dbh["RMSE"],
            "DBH_MAE": dbh["MAE"],
            "DBH_bias": dbh["bias"],
            "DBH_R2": dbh["R2"],
        }
    )


# Combine the recalculated results into one row per scanner
challenging_recalculated_summary = pd.DataFrame(
    challenging_summary_rows
)

challenging_recalculated_summary


# Combine the earlier scanner summaries into one table and standardise their scanner labels

old_challenging_summary = pd.concat(
    challenging_validation_summaries.values(),
    ignore_index=True,
)


old_challenging_summary["Scanner"] = (
    old_challenging_summary["scanner"]
    .map(SCANNER_NAME_MAP)
)


# Check that every scanner name in the earlier summaries was mapped successfully
unmapped_summary_scanners = old_challenging_summary.loc[
    old_challenging_summary["Scanner"].isna(),
    "scanner"
].dropna().unique()


require(
    len(unmapped_summary_scanners) == 0,
    "Unrecognised scanner labels in the Challenging summaries: "
    f"{sorted(unmapped_summary_scanners)}"
)


# Compare the recalculated results with the earlier summaries

metric_columns = [
    "TH_n",
    "TH_RMSE",
    "TH_MAE",
    "TH_bias",
    "TH_R2",
    "DBH_n",
    "DBH_RMSE",
    "DBH_MAE",
    "DBH_bias",
    "DBH_R2",
]


# Join the two summaries by scanner so each metric can be compared directly
challenging_summary_check = (
    challenging_recalculated_summary.merge(
        old_challenging_summary[
            ["Scanner"] + metric_columns
        ],
        on="Scanner",
        suffixes=("_new", "_old"),
        validate="one_to_one",
    )
)


# Check that every recalculated metric matches the value stored in the earlier summary files within a very small tolerance
for metric in metric_columns:

    require(
        np.allclose(
            challenging_summary_check[f"{metric}_new"],
            challenging_summary_check[f"{metric}_old"],
            rtol=1e-8,
            atol=1e-10,
        ),
        f"Challenging validation failed for {metric}."
    )


print(
    "Challenging summary validation passed: the recalculated "
    "post-QC statistics match the earlier summary files."
)

Challenging summary validation passed: the recalculated post-QC statistics match the earlier summary files.


# Combine the RCT measurement data

The Easy, Intermediate and Challenging datasets have now been reconstructed separately and checked against the outputs from the original analysis.

They are combined here into one consistent per-tree dataset. Only the final one-to-one GT matches used for TH and DBH accuracy are included.

In [35]:
# Combine the three terrain-level measurement tables
# Keep only fields with the same meaning across Easy, Intermediate and Challenging
# So the three datasets can be combined into one consistent RCT measurement table
# Raw dataset names are not needed here because Terrain + Scanner + segment_id identifies each original RCT segment
# Intermediate contains the cleaned .ply filename for trees that were reprocessed
# Easy and Challenging do not have an equivalent field, so it is left blank for those terrains

easy_measurements["source_clean_file"] = pd.NA
challenging_measurements["source_clean_file"] = pd.NA


rct_before_after = pd.concat(
    [
        easy_measurements,
        intermediate_measurements,
        challenging_measurements,
    ],
    ignore_index=True,
)


# Standardise the GT identifiers
# All records in the combined table are final one-to-one matches,so each GT_ID should contain a single numeric tree ID

rct_before_after["GT_ID"] = pd.to_numeric(
    rct_before_after["GT_ID"],
    errors="raise",
)


# Check that all numeric GT IDs are whole numbers before converting them to integer format
require(
    np.allclose(
        rct_before_after["GT_ID"],
        np.round(rct_before_after["GT_ID"]),
    ),
    "A non-integer GT identifier was found in the final RCT table."
)


rct_before_after["GT_ID"] = (
    rct_before_after["GT_ID"]
    .astype(int)
)


# Set the final column order used for the combined RCT dataset

final_rct_columns = [
    "Terrain",
    "Scanner",
    "segment_id",
    "GT_ID",

    "GT_TH",
    "before_TH",
    "after_TH",
    "before_TH_error",
    "after_TH_error",
    "before_TH_abs_error",
    "after_TH_abs_error",
    "TH_delta_abs_error",

    "GT_DBH",
    "before_DBH",
    "after_DBH",
    "before_DBH_error",
    "after_DBH_error",
    "before_DBH_abs_error",
    "after_DBH_abs_error",
    "DBH_delta_abs_error",

    "attribute_value_source",
    "source_clean_file",
]


rct_before_after = rct_before_after[
    final_rct_columns
].copy()


# Check the final combined dataset
# The three terrain tables should contain 150 scanner-tree records in total
require(
    len(rct_before_after) == 150,
    "Expected 150 RCT scanner-tree records in total."
)


# Each RCT segment should appear only once within each terrain and scanner
require(
    not rct_before_after.duplicated(
        subset=["Terrain", "Scanner", "segment_id"]
    ).any(),
    "Duplicate Terrain + Scanner + segment_id records found."
)


# Each retained GT tree should also appear only once for each scanner within a terrain
require(
    not rct_before_after.duplicated(
        subset=["Terrain", "Scanner", "GT_ID"]
    ).any(),
    "Duplicate Terrain + Scanner + GT_ID records found."
)


# RCT_tree_id is no longer used and should not appear in the cleaned combined dataset
require(
    "RCT_tree_id" not in rct_before_after.columns,
    "RCT_tree_id should not be present in the cleaned dataset."
)


print("Combined RCT measurement table created successfully.")
print("Rows:", len(rct_before_after))

print("\nRows by terrain and scanner:")
print(
    rct_before_after
    .groupby(["Terrain", "Scanner"])
    .size()
)

Combined RCT measurement table created successfully.
Rows: 150

Rows by terrain and scanner:
Terrain       Scanner      
Challenging   Dual sensor      19
              Hovermap         19
              Single sensor    19
Easy          Dual sensor       9
              Hovermap          9
              Single sensor     9
Intermediate  Dual sensor      22
              Hovermap         22
              Single sensor    22
dtype: int64


In [36]:
# Check that all stored error values can be reproduced directly from the GT and RCT measurements
# This makes sure that no older error value has been carried forward after a measurement was updated during QC

require(
    np.allclose(
        rct_before_after["before_TH_error"],
        rct_before_after["before_TH"] - rct_before_after["GT_TH"],
    ),
    "Before-QC TH errors are inconsistent."
)

require(
    np.allclose(
        rct_before_after["after_TH_error"],
        rct_before_after["after_TH"] - rct_before_after["GT_TH"],
    ),
    "After-QC TH errors are inconsistent."
)

require(
    np.allclose(
        rct_before_after["before_DBH_error"],
        rct_before_after["before_DBH"] - rct_before_after["GT_DBH"],
    ),
    "Before-QC DBH errors are inconsistent."
)

require(
    np.allclose(
        rct_before_after["after_DBH_error"],
        rct_before_after["after_DBH"] - rct_before_after["GT_DBH"],
    ),
    "After-QC DBH errors are inconsistent."
)


# Check that the recorded change in absolute error is also consistent with the before- and after-QC error values
require(
    np.allclose(
        rct_before_after["TH_delta_abs_error"],
        rct_before_after["after_TH_error"].abs()
        - rct_before_after["before_TH_error"].abs(),
    ),
    "TH change in absolute error is inconsistent."
)

require(
    np.allclose(
        rct_before_after["DBH_delta_abs_error"],
        rct_before_after["after_DBH_error"].abs()
        - rct_before_after["before_DBH_error"].abs(),
    ),
    "DBH change in absolute error is inconsistent."
)


print("All combined RCT error checks passed.")

All combined RCT error checks passed.


In [37]:
# Save the final analysis-ready RCT measurement dataset
# The original source files are left unchanged
# The cleaned combined dataset is saved as a new processed file for use in the later analysis notebooks

RCT_PROCESSED_DIR = PROCESSED_DIR / "rct"
RCT_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


RCT_BEFORE_AFTER_FILE = (
    RCT_PROCESSED_DIR / "rct_before_after_per_tree.csv"
)


rct_before_after.to_csv(
    RCT_BEFORE_AFTER_FILE,
    index=False,
)


print(
    "Saved:",
    RCT_BEFORE_AFTER_FILE.relative_to(PROJECT_DIR)
)

Saved: data/processed/rct/rct_before_after_per_tree.csv


In [38]:
# Read the saved RCT dataset back in as a final export check
# This confirms that the file can be loaded independently and still contains the expected number of rows and columns

rct_export_check = pd.read_csv(
    RCT_BEFORE_AFTER_FILE
)


require(
    len(rct_export_check) == 150,
    "The exported RCT dataset does not contain 150 rows."
)

require(
    rct_export_check.columns.tolist()
    == rct_before_after.columns.tolist(),
    "The exported RCT dataset has unexpected columns."
)


print(
    "Export check passed:",
    len(rct_export_check),
    "rows and",
    len(rct_export_check.columns),
    "columns."
)

Export check passed: 150 rows and 22 columns.


# RCT candidate and detection QC

This section prepares the candidate-level RCT data used for the tree-detection analysis.

Unlike the measurement table, which contains only the final one-to-one GT matches, this table keeps all RCT candidates. This means valid detections, genuine unrecorded trees, fragments and under-segmented cases are all retained for the detection analysis.

In [39]:
# Inspect the Easy candidate and manual-QC information
# Before building the final detection table:
# 1. check how the original 31 RCT candidates were classified 
# 2. and which segments later received an explicit manual-QC decision
# This keeps the final detection classifications traceable to the original candidate and QC records

print("Easy candidate statuses:")
print(
    easy_candidate_qc["candidate_status"]
    .value_counts(dropna=False)
)


print("\nManual QC classes:")
print(
    easy_manual_qc["QC_class"]
    .value_counts(dropna=False)
)


print("\nManual QC decisions:")
display(
    easy_manual_qc[
        [
            "dataset",
            "segment_id",
            "GT_ID_final",
            "QC_class",
            "include_in_GT_detection",
            "include_in_attribute_accuracy",
            "QC_notes",
        ]
    ].sort_values(
        ["dataset", "segment_id"]
    )
)


# Check that every manual-QC record refers to a segment that exists in the original Easy candidate table

candidate_keys = set(
    easy_candidate_qc[
        ["dataset", "segment_id"]
    ].itertuples(index=False, name=None)
)

manual_keys = set(
    easy_manual_qc[
        ["dataset", "segment_id"]
    ].itertuples(index=False, name=None)
)


# Any remaining key would indicate a QC decision that cannot be linked back to an original RCT candidate
missing_manual_candidates = manual_keys - candidate_keys

require(
    len(missing_manual_candidates) == 0,
    "At least one Easy manual-QC decision refers to a candidate "
    "that is not present in the original candidate table."
)


print("\nAll Easy manual-QC decisions match original candidates.")

Easy candidate statuses:
assigned_to_GT             27
unmatched_RCT_candidate     4
Name: candidate_status, dtype: int64

Manual QC classes:
validated_primary_match    6
genuine_unrecorded_tree    3
split_fragment             1
Name: QC_class, dtype: int64

Manual QC decisions:


,dataset,segment_id,GT_ID_final,QC_class,include_in_GT_detection,include_in_attribute_accuracy,QC_notes
7,et_dvojskener_nogrid,0,21.0,validated_primary_match,Yes,Yes,Primary segment corresponding to GT 21
4,et_dvojskener_nogrid,5,NaN,genuine_unrecorded_tree,No,No,Genuine small tree visible in the point cloud ...
1,et_dvojskener_nogrid,7,33.0,validated_primary_match,Yes,Yes,Clear corresponding tree in CloudCompare despi...
9,et_hovermap_nogrid,0,21.0,split_fragment,No,No,Small fragment belonging to Hovermap segment 1...
8,et_hovermap_nogrid,1,21.0,validated_primary_match,Yes,Yes,Primary Hovermap segment corresponding to GT 21
5,et_hovermap_nogrid,6,NaN,genuine_unrecorded_tree,No,No,Genuine small tree visible in the point cloud ...
2,et_hovermap_nogrid,8,33.0,validated_primary_match,Yes,Yes,Clear corresponding tree in CloudCompare despi...
6,et_j_nogrid,0,21.0,validated_primary_match,Yes,Yes,Primary segment corresponding to GT 21
3,et_j_nogrid,5,NaN,genuine_unrecorded_tree,No,No,Genuine small tree visible in the point cloud ...
0,et_j_nogrid,7,33.0,validated_primary_match,Yes,Yes,Clear corresponding tree in CloudCompare despi...



All Easy manual-QC decisions match original candidates.


In [40]:
# Build the final Easy candidate-level QC table
# Start with all 31 original RCT candidates and add the manual-QC decisions where these are available
# Candidates already matched clearly to a GT tree are kept as validated primary matches
# unless a manual decision says otherwise
# All unmatched candidates should be explained by the manual-QC table

easy_detection = easy_candidates.copy()


# Check that every unmatched candidate has an explicit manual-QC decision

easy_unmatched = easy_detection.loc[
    easy_detection["candidate_status"] == "unmatched_RCT_candidate",
    ["dataset", "segment_id"]
]


easy_manual_keys = set(
    easy_manual_qc[
        ["dataset", "segment_id"]
    ].itertuples(index=False, name=None)
)

easy_unmatched_keys = set(
    easy_unmatched.itertuples(index=False, name=None)
)


require(
    easy_unmatched_keys.issubset(easy_manual_keys),
    "At least one unmatched Easy candidate has no manual QC decision."
)


print(
    "Unmatched candidates:",
    len(easy_unmatched),
    "- all have manual QC decisions."
)


# Prepare the manual-QC decisions for merging
# Some older QC files store inclusion decisions as Yes/No rather than True/False
# So these values are converted to a consistent boolean format

def to_boolean(value):
    """Convert common Yes/No and True/False values to booleans."""

    if pd.isna(value):
        return pd.NA

    if isinstance(value, (bool, np.bool_)):
        return bool(value)

    text = str(value).strip().lower()

    if text in {"yes", "true", "1"}:
        return True

    if text in {"no", "false", "0"}:
        return False

    raise ValueError(f"Unrecognised boolean value: {value}")


easy_manual = easy_manual_qc.copy()

easy_manual["include_in_GT_detection"] = (
    easy_manual["include_in_GT_detection"]
    .map(to_boolean)
)

easy_manual["include_in_attribute_accuracy"] = (
    easy_manual["include_in_attribute_accuracy"]
    .map(to_boolean)
)


# Rename the manual-QC fields so they remain separate from the original candidate fields after the tables are joined
easy_manual = easy_manual.rename(
    columns={
        "GT_ID_final": "manual_GT_ID_final",
        "QC_class": "manual_QC_class",
        "include_in_GT_detection":
            "manual_include_in_GT_detection",
        "include_in_attribute_accuracy":
            "manual_include_in_attribute_accuracy",
        "QC_notes": "manual_QC_notes",
    }
)


# Add the manual-QC information to the complete candidate table

easy_detection = easy_detection.merge(
    easy_manual[
        [
            "dataset",
            "segment_id",
            "manual_GT_ID_final",
            "manual_QC_class",
            "manual_include_in_GT_detection",
            "manual_include_in_attribute_accuracy",
            "manual_QC_notes",
        ]
    ],
    on=["dataset", "segment_id"],
    how="left",
    validate="one_to_one",
)


# Create the final Easy QC fields
# Where a manual decision exists, it is used first
# Otherwise, candidates originally assigned to a GT tree are treated as validated one-to-one matches

easy_detection["GT_ID_final"] = (
    easy_detection["manual_GT_ID_final"]
    .combine_first(easy_detection["assigned_GT_ID"])
)


easy_detection["QC_class"] = (
    easy_detection["manual_QC_class"]
)


easy_detection.loc[
    easy_detection["QC_class"].isna()
    & easy_detection["candidate_status"].eq("assigned_to_GT"),
    "QC_class"
] = "validated_primary_match"


# Define detection and attribute-accuracy inclusion
# A validated primary match represents one GT tree recovered as one independent RCT tree and receives one detection
# Genuine unrecorded trees are real trees but do not receive GT detection credit
# Split fragments are not treated as independent tree detections

easy_detection["GT_detection_credit"] = np.where(
    easy_detection["QC_class"].eq("validated_primary_match"),
    1,
    0,
)


easy_detection["include_in_attribute_accuracy"] = (
    easy_detection["QC_class"].eq("validated_primary_match")
)


easy_detection["is_genuine_unrecorded_tree"] = (
    easy_detection["QC_class"].eq("genuine_unrecorded_tree")
)


# Check the final Easy candidate classifications

require(
    easy_detection["QC_class"].notna().all(),
    "At least one Easy candidate has no final QC classification."
)


require(
    len(easy_detection) == 31,
    "Expected 31 candidates in the final Easy candidate table."
)


require(
    easy_detection["GT_detection_credit"].sum() == 27,
    "Expected 27 valid GT detections across the three Easy datasets."
)


require(
    easy_detection["include_in_attribute_accuracy"].sum() == 27,
    "Expected 27 Easy candidates to be retained for attribute accuracy."
)


require(
    easy_detection["is_genuine_unrecorded_tree"].sum() == 3,
    "Expected three genuine unrecorded Easy trees."
)


print("Final Easy QC classes:")
print(
    easy_detection["QC_class"]
    .value_counts()
)

print("\nGT detections by scanner:")
print(
    easy_detection
    .groupby("Scanner")["GT_detection_credit"]
    .sum()
)

print("\nGenuine unrecorded trees by scanner:")
print(
    easy_detection
    .groupby("Scanner")["is_genuine_unrecorded_tree"]
    .apply(lambda x: x.astype(int).sum())
)

Unmatched candidates: 4 - all have manual QC decisions.
Final Easy QC classes:
validated_primary_match    27
genuine_unrecorded_tree     3
split_fragment              1
Name: QC_class, dtype: int64

GT detections by scanner:
Scanner
Dual sensor      9
Hovermap         9
Single sensor    9
Name: GT_detection_credit, dtype: int64

Genuine unrecorded trees by scanner:
Scanner
Dual sensor      1
Hovermap         1
Single sensor    1
Name: is_genuine_unrecorded_tree, dtype: int64


In [41]:
# Cross-check the Easy detection and measurement samples
# Every candidate retained for attribute accuracy should also appear in the final Easy before/after measurement table
# This confirms that the detection QC and measurement analysis are using the same set of one-to-one RCT tree segments

easy_detection_matches = set(
    easy_detection.loc[
        easy_detection["include_in_attribute_accuracy"],
        ["Scanner", "segment_id"]
    ].itertuples(index=False, name=None)
)


easy_measurement_matches = set(
    easy_measurements[
        ["Scanner", "segment_id"]
    ].itertuples(index=False, name=None)
)


require(
    easy_detection_matches == easy_measurement_matches,
    "The Easy detection and attribute-accuracy samples do not match."
)


print(
    "Easy detection/measurement cross-check passed:",
    len(easy_detection_matches),
    "one-to-one candidate records."
)

Easy detection/measurement cross-check passed: 27 one-to-one candidate records.


In [42]:
# Inspect the Intermediate candidate and manual-QC information
# The Intermediate candidate table contains 72 RCT segments
# Inspect the original candidate classifications and the six manual-QC decisions before building the final table

print("Intermediate candidate statuses:")
print(
    intermediate_candidates_before["candidate_status"]
    .value_counts(dropna=False)
)


print("\nManual QC classes:")
print(
    intermediate_manual_qc["QC_class"]
    .value_counts(dropna=False)
)


print("\nManual QC decisions:")
display(
    intermediate_manual_qc[
        [
            "dataset",
            "segment_id",
            "GT_ID_final",
            "QC_class",
            "include_in_GT_detection",
            "include_in_attribute_accuracy",
            "QC_notes",
        ]
    ].sort_values(
        ["dataset", "segment_id"]
    )
)


# Check that every unmatched candidate has an explicit manual-QC decision

intermediate_unmatched = intermediate_candidates_before.loc[
    intermediate_candidates_before["candidate_status"]
    == "unmatched_RCT_candidate",
    ["dataset", "segment_id"]
]


intermediate_manual_keys = set(
    intermediate_manual_qc[
        ["dataset", "segment_id"]
    ].itertuples(index=False, name=None)
)

intermediate_unmatched_keys = set(
    intermediate_unmatched.itertuples(index=False, name=None)
)


require(
    intermediate_unmatched_keys.issubset(intermediate_manual_keys),
    "At least one unmatched Intermediate candidate has no "
    "manual QC decision."
)


print(
    "\nUnmatched candidates:",
    len(intermediate_unmatched),
    "- all have manual QC decisions."
)

Intermediate candidate statuses:
assigned_to_GT             66
unmatched_RCT_candidate     6
Name: candidate_status, dtype: int64

Manual QC classes:
genuine_unrecorded_tree        3
genuine_edge_truncated_tree    3
Name: QC_class, dtype: int64

Manual QC decisions:


,dataset,segment_id,GT_ID_final,QC_class,include_in_GT_detection,include_in_attribute_accuracy,QC_notes
1,it_dvojskener_nogrid,11,NaN,genuine_unrecorded_tree,No,No,Same distinct small tree detected by Dvojskenn...
4,it_dvojskener_nogrid,21,NaN,genuine_edge_truncated_tree,No,No,Same edge-truncated tree detected by Dvojskenn...
2,it_hovermap_nogrid,10,NaN,genuine_unrecorded_tree,No,No,Same distinct small tree detected by Hovermap;...
5,it_hovermap_nogrid,21,NaN,genuine_edge_truncated_tree,No,No,Same edge-truncated tree detected by Hovermap;...
0,it_jednoskener_nogrid,10,NaN,genuine_unrecorded_tree,No,No,Distinct small tree with its own stem; absent ...
3,it_jednoskener_nogrid,21,NaN,genuine_edge_truncated_tree,No,No,Distinct tree standing at the terrain edge and...



Unmatched candidates: 6 - all have manual QC decisions.


In [43]:
# Build the final Intermediate candidate-level QC table
# Start with all 72 original RCT candidates and add the six manual-QC decisions where these are available
# Candidates already matched clearly to a GT tree are kept as validated primary matches
# unless a manual decision says otherwise.

intermediate_detection = intermediate_candidates_pre.copy()


# Convert the Yes/No fields in the manual-QC file to a consistent boolean format

intermediate_manual = intermediate_manual_qc.copy()

intermediate_manual["include_in_GT_detection"] = (
    intermediate_manual["include_in_GT_detection"]
    .map(to_boolean)
)

intermediate_manual["include_in_attribute_accuracy"] = (
    intermediate_manual["include_in_attribute_accuracy"]
    .map(to_boolean)
)


# Rename the manual-QC fields so they remain separate from the original candidate information after the tables are joined

intermediate_manual = intermediate_manual.rename(
    columns={
        "GT_ID_final": "manual_GT_ID_final",
        "QC_class": "manual_QC_class",
        "include_in_GT_detection":
            "manual_include_in_GT_detection",
        "include_in_attribute_accuracy":
            "manual_include_in_attribute_accuracy",
        "QC_notes": "manual_QC_notes",
    }
)


# Add the manual-QC decisions to the complete candidate table

intermediate_detection = intermediate_detection.merge(
    intermediate_manual[
        [
            "dataset",
            "segment_id",
            "manual_GT_ID_final",
            "manual_QC_class",
            "manual_include_in_GT_detection",
            "manual_include_in_attribute_accuracy",
            "manual_QC_notes",
        ]
    ],
    on=["dataset", "segment_id"],
    how="left",
    validate="one_to_one",
)


# Create the final Intermediate QC fields
# Manual decisions are used where they exist
# All remaining candidates that were already assigned to a GT tree are treated as validated one-to-one matches

intermediate_detection["GT_ID_final"] = (
    intermediate_detection["manual_GT_ID_final"]
    .combine_first(intermediate_detection["assigned_GT_ID"])
)


intermediate_detection["QC_class"] = (
    intermediate_detection["manual_QC_class"]
)


intermediate_detection.loc[
    intermediate_detection["QC_class"].isna()
    & intermediate_detection["candidate_status"].eq("assigned_to_GT"),
    "QC_class"
] = "validated_primary_match"


# Define the final detection and attribute-accuracy fields
# Only validated one-to-one GT matches receive GT detection credit and are included in the TH/DBH accuracy analysis
# Genuine unrecorded and edge-truncated trees remain in the candidate table because they are real RCT detections
# but they are not counted as detections of reference GT trees

intermediate_detection["GT_detection_credit"] = np.where(
    intermediate_detection["QC_class"].eq("validated_primary_match"),
    1,
    0,
)


intermediate_detection["include_in_attribute_accuracy"] = (
    intermediate_detection["QC_class"]
    .eq("validated_primary_match")
)


intermediate_detection["is_genuine_unrecorded_tree"] = (
    intermediate_detection["QC_class"]
    .eq("genuine_unrecorded_tree")
)


intermediate_detection["is_genuine_edge_truncated_tree"] = (
    intermediate_detection["QC_class"]
    .eq("genuine_edge_truncated_tree")
)


# Check the final Intermediate candidate classifications.

require(
    intermediate_detection["QC_class"].notna().all(),
    "At least one Intermediate candidate has no final QC classification."
)


require(
    len(intermediate_detection) == 72,
    "Expected 72 candidates in the Intermediate candidate table."
)


require(
    intermediate_detection["GT_detection_credit"].sum() == 66,
    "Expected 66 valid GT detections across the three Intermediate datasets."
)


require(
    intermediate_detection["include_in_attribute_accuracy"].sum() == 66,
    "Expected 66 Intermediate candidates to enter attribute accuracy."
)


require(
    intermediate_detection["is_genuine_unrecorded_tree"].sum() == 3,
    "Expected three genuine unrecorded Intermediate trees."
)


require(
    intermediate_detection["is_genuine_edge_truncated_tree"].sum() == 3,
    "Expected three genuine edge-truncated Intermediate trees."
)


print("Final Intermediate QC classes:")
print(
    intermediate_detection["QC_class"]
    .value_counts()
)

print("\nGT detections by scanner:")
print(
    intermediate_detection
    .groupby("Scanner")["GT_detection_credit"]
    .sum()
)

print("\nGenuine unrecorded trees by scanner:")
print(
    intermediate_detection
    .groupby("Scanner")["is_genuine_unrecorded_tree"]
    .apply(lambda x: x.astype(int).sum())
)

print("\nGenuine edge-truncated trees by scanner:")
print(
    intermediate_detection
    .groupby("Scanner")["is_genuine_edge_truncated_tree"]
    .apply(lambda x: x.astype(int).sum())
)

Final Intermediate QC classes:
validated_primary_match        66
genuine_unrecorded_tree         3
genuine_edge_truncated_tree     3
Name: QC_class, dtype: int64

GT detections by scanner:
Scanner
Dual sensor      22
Hovermap         22
Single sensor    22
Name: GT_detection_credit, dtype: int64

Genuine unrecorded trees by scanner:
Scanner
Dual sensor      1
Hovermap         1
Single sensor    1
Name: is_genuine_unrecorded_tree, dtype: int64

Genuine edge-truncated trees by scanner:
Scanner
Dual sensor      1
Hovermap         1
Single sensor    1
Name: is_genuine_edge_truncated_tree, dtype: int64


In [44]:
# Cross-check the Intermediate detection and measurement samples
# Every candidate retained for attribute accuracy should also appear in the final Intermediate before/after measurement table
# This confirms that the detection QC and measurement analysis are using the same set of one-to-one RCT tree segments

intermediate_detection_matches = set(
    intermediate_detection.loc[
        intermediate_detection["include_in_attribute_accuracy"],
        ["Scanner", "segment_id"]
    ].itertuples(index=False, name=None)
)


intermediate_measurement_matches = set(
    intermediate_measurements[
        ["Scanner", "segment_id"]
    ].itertuples(index=False, name=None)
)


require(
    intermediate_detection_matches
    == intermediate_measurement_matches,
    "The Intermediate detection and measurement samples do not match."
)


print(
    "Intermediate detection/measurement cross-check passed:",
    len(intermediate_detection_matches),
    "one-to-one candidate records."
)

Intermediate detection/measurement cross-check passed: 66 one-to-one candidate records.


In [45]:
# Validate the Intermediate candidate classifications
# The v2 candidate table comes from the earlier analysis 
# It is used here only to check that the rebuilt classifications match the final validated output
# It does not contribute to the new candidate classifications

intermediate_v2_candidates_check = intermediate_v2_candidates.copy()


# Standardise the scanner labels in the validation table so they match the labels used in the cleaned workflow

intermediate_v2_candidates_check["Scanner"] = (
    intermediate_v2_candidates_check["scanner"]
    .map(SCANNER_NAME_MAP)
)


require(
    intermediate_v2_candidates_check["Scanner"].notna().all(),
    "An unknown scanner label was found in the Intermediate "
    "validation candidate table."
)


# Join the rebuilt and earlier candidate tables using Scanner + segment_id

candidate_validation = intermediate_detection.merge(
    intermediate_v2_candidates_check[
        [
            "Scanner",
            "segment_id",
            "GT_ID_final",
            "QC_class",
            "include_in_attribute_accuracy",
        ]
    ],
    on=["Scanner", "segment_id"],
    how="inner",
    suffixes=("_new", "_old"),
    validate="one_to_one",
)


require(
    len(candidate_validation) == 72,
    "Expected all 72 Intermediate candidates to match the "
    "earlier validation table."
)


# Compare the final QC classes
# The earlier v2 table uses 'accepted_automatic_primary_match' for the retained one-to-one GT matches
# The cleaned workflow uses 'validated_primary_match' for the same type of record
# Standardise this older label before comparing the two versions

historical_qc_class = (
    candidate_validation["QC_class_old"]
    .replace(
        {
            "accepted_automatic_primary_match":
                "validated_primary_match"
        }
    )
)


require(
    (
        candidate_validation["QC_class_new"]
        == historical_qc_class
    ).all(),
    "Intermediate QC classes do not match after standardising "
    "the earlier class labels."
)


# Compare the final GT IDs

require(
    np.allclose(
        pd.to_numeric(
            candidate_validation["GT_ID_final_new"],
            errors="coerce",
        ),
        pd.to_numeric(
            candidate_validation["GT_ID_final_old"],
            errors="coerce",
        ),
        equal_nan=True,
    ),
    "Intermediate final GT IDs do not match the earlier v2 output."
)


# Compare which candidates are retained for attribute accuracy

require(
    (
        candidate_validation["include_in_attribute_accuracy_new"]
        .astype(bool)
        ==
        candidate_validation["include_in_attribute_accuracy_old"]
        .astype(bool)
    ).all(),
    "Intermediate attribute-accuracy inclusion does not match "
    "the earlier v2 output."
)


print(
    "Intermediate candidate validation passed: all 72 candidate "
    "classifications match the earlier v2 output."
)

Intermediate candidate validation passed: all 72 candidate classifications match the earlier v2 output.


In [46]:
# Standardise the Challenging candidate classifications
# The three scanner-specific QC files use different status labels
# Keep the original status in source_QC_status
# then map these to one consistent QC_class_final field for the combined analysis.
# This keeps the original manual interpretation visible while allowing the same QC categories to be used across all scanners


# Single sensor
challenging_single_detection = challenging_single_qc.copy()

challenging_single_detection["Terrain"] = "Challenging"
challenging_single_detection["Scanner"] = "Single sensor"

challenging_single_detection["source_QC_status"] = (
    challenging_single_detection["QC_class"]
)


# Map the original Single-sensor classes to the common final labels

single_class_map = {
    "validated_primary_match":
        "validated_primary_match",

    "validated_primary_match_after_manual_split":
        "validated_primary_match",

    "merged_GT_trees":
        "undersegmented_GT_trees",

    "split_fragment":
        "invalid_fragment",

    "rejected_non_tree_segment":
        "invalid_non_tree_segment",

    "genuine_unrecorded_tree":
        "genuine_unrecorded_tree",

    "genuine_unrecorded_fallen_tree":
        "genuine_unrecorded_fallen_tree",
}


challenging_single_detection["QC_class_final"] = (
    challenging_single_detection["QC_class"]
    .map(single_class_map)
)


# Check that every original Single-sensor class was mapped
require(
    challenging_single_detection["QC_class_final"].notna().all(),
    "An unknown Single-sensor Challenging QC class was found."
)


# Dual sensor
challenging_dual_detection = challenging_dual_qc.copy()

challenging_dual_detection["Terrain"] = "Challenging"
challenging_dual_detection["Scanner"] = "Dual sensor"

challenging_dual_detection["source_QC_status"] = (
    challenging_dual_detection["final_status"]
)


# Map the original Dual-sensor status labels to the common classes

dual_class_map = {
    "cleaned_retained_tree":
        "validated_primary_match",

    "validated_untouched_tree":
        "validated_primary_match",

    "cleaned_reprocessing_exception":
        "validated_primary_match",

    "validated_primary_match":
        "validated_primary_match",

    "mixed_undersegmented":
        "undersegmented_GT_trees",

    "ambiguous_undersegmented":
        "undersegmented_GT_trees",

    "random_or_unresolved_fragment":
        "invalid_fragment",

    "genuine_unrecorded_tree":
        "genuine_unrecorded_tree",
}


challenging_dual_detection["QC_class_final"] = (
    challenging_dual_detection["final_status"]
    .map(dual_class_map)
)


# Check that every original Dual-sensor status was mapped
require(
    challenging_dual_detection["QC_class_final"].notna().all(),
    "An unknown Dual-sensor Challenging QC class was found."
)


# Hovermap
challenging_hovermap_detection = challenging_hovermap_qc.copy()

challenging_hovermap_detection["Terrain"] = "Challenging"
challenging_hovermap_detection["Scanner"] = "Hovermap"

challenging_hovermap_detection["source_QC_status"] = (
    challenging_hovermap_detection["final_status"]
)


# Map the original Hovermap status labels to the common classes

hovermap_class_map = {
    "cleaned_retained_tree":
        "validated_primary_match",

    "validated_untouched_tree":
        "validated_primary_match",

    "cleaned_reprocessing_exception":
        "validated_primary_match",

    "validated_primary_match":
        "validated_primary_match",

    # This case is kept as a valid one-to-one GT match even though the neighbouring stems required additional manual checking
    "ambiguous_close_neighbours":
        "validated_primary_match",

    "mixed_undersegmented":
        "undersegmented_GT_trees",

    "ambiguous_undersegmented":
        "undersegmented_GT_trees",

    "random_or_unresolved_fragment":
        "invalid_fragment",

    "absorbed_fragment":
        "invalid_fragment",

    "segmentation_unreliable":
        "invalid_fragment",

    "genuine_unrecorded_tree":
        "genuine_unrecorded_tree",
}


challenging_hovermap_detection["QC_class_final"] = (
    challenging_hovermap_detection["final_status"]
    .map(hovermap_class_map)
)


# Check that every original Hovermap status was mapped
require(
    challenging_hovermap_detection["QC_class_final"].notna().all(),
    "An unknown Hovermap Challenging QC class was found."
)


print("All Challenging QC status labels were standardised.")

All Challenging QC status labels were standardised.


In [47]:
# Create common Challenging detection fields
# Derive the final detection and inclusion fields from the standardised QC classes rather than copying the older flags
# This keeps the final detection rules consistent across all three scanners, especially for merged and under-segmented cases

for df in [
    challenging_single_detection,
    challenging_dual_detection,
    challenging_hovermap_detection,
]:

    # Give detection credit only when one RCT segment represents one valid reference tree
    df["GT_detection_credit"] = np.where(
        df["QC_class_final"].eq("validated_primary_match"),
        1,
        0,
    )

    # Only valid one-to-one GT matches are included in the TH/DBH accuracy analysis
    df["include_in_attribute_accuracy_final"] = (
        df["QC_class_final"]
        .eq("validated_primary_match")
    )

    # Keep genuine trees that are absent from the reference data identifiable for later detection-correctness calculations
    df["is_genuine_unrecorded_tree"] = (
        df["QC_class_final"].isin(
            [
                "genuine_unrecorded_tree",
                "genuine_unrecorded_fallen_tree",
            ]
        )
    )

    # Mark under-segmented candidates separately because they remain detection errors at the individual-tree level
    df["is_undersegmented"] = (
        df["QC_class_final"]
        .eq("undersegmented_GT_trees")
    )

    # Mark fragments and non-tree segments as invalid candidates
    df["is_invalid_candidate"] = (
        df["QC_class_final"].isin(
            [
                "invalid_fragment",
                "invalid_non_tree_segment",
            ]
        )
    )


# Combine the three scanner-specific QC tables into one Challenging-terrain candidate table

challenging_detection = pd.concat(
    [
        challenging_single_detection,
        challenging_dual_detection,
        challenging_hovermap_detection,
    ],
    ignore_index=True,
)


# Check the final Challenging candidate classifications

require(
    len(challenging_detection) == 128,
    "Expected 128 Challenging RCT candidates."
)


require(
    challenging_detection["QC_class_final"].notna().all(),
    "At least one Challenging candidate has no final QC class."
)


print("Final Challenging QC classes:")
print(
    challenging_detection["QC_class_final"]
    .value_counts()
)


print("\nGT detection credit by scanner:")
print(
    challenging_detection
    .groupby("Scanner")["GT_detection_credit"]
    .sum()
)


print("\nCandidates retained for attribute accuracy:")
print(
    challenging_detection
    .groupby("Scanner")["include_in_attribute_accuracy_final"]
    .apply(lambda x: x.astype(int).sum())
)

Final Challenging QC classes:
validated_primary_match           57
genuine_unrecorded_tree           31
invalid_fragment                  17
undersegmented_GT_trees           12
invalid_non_tree_segment          10
genuine_unrecorded_fallen_tree     1
Name: QC_class_final, dtype: int64

GT detection credit by scanner:
Scanner
Dual sensor      19
Hovermap         19
Single sensor    19
Name: GT_detection_credit, dtype: int64

Candidates retained for attribute accuracy:
Scanner
Dual sensor      19
Hovermap         19
Single sensor    19
Name: include_in_attribute_accuracy_final, dtype: int64


In [48]:
# Cross-check the Challenging measurement sample
# Every tree included in the TH/DBH measurement table
# should also be classified as a valid one-to-one GT match in the candidate QC
# The two sets do not need to be identical as some valid detection records may still be excluded from measurement analysis

challenging_detection_measurement_ids = set(
    challenging_detection.loc[
        challenging_detection[
            "include_in_attribute_accuracy_final"
        ],
        ["Scanner", "segment_id"]
    ].itertuples(index=False, name=None)
)


challenging_measurement_ids = set(
    challenging_measurements[
        ["Scanner", "segment_id"]
    ].itertuples(index=False, name=None)
)


# Identify any measurement records that do not have a matching valid candidate-level QC classification
missing_measurement_candidates = (
    challenging_measurement_ids
    - challenging_detection_measurement_ids
)


require(
    len(missing_measurement_candidates) == 0,
    "At least one Challenging measurement record is not classified "
    "as a valid one-to-one candidate."
)


print(
    "All 57 Challenging measurement records match valid "
    "candidate-level QC records."
)

print(
    "\nCandidate records classified for attribute accuracy:",
    len(challenging_detection_measurement_ids)
)

All 57 Challenging measurement records match valid candidate-level QC records.

Candidate records classified for attribute accuracy: 57


In [49]:
# Put the Challenging candidate data into one consistent format
# The three scanner-specific QC tables use slightly different fields
# so the information needed for the final detection analysis is standardised here
# The GT tree(s) represented by each candidate and the original QC status are kept so that difficult cases can still be traced


def standardise_gt_ids(value):
    """
    Convert the different GT-ID formats to one consistent format.

    Examples:
        6 cenverted into "6"
        "GT6"converted into "6"
        "GT23/GT24" converted into"23;24"
        "GT35+GT36" converted into "35;36"

    Missing values are left as missing.
    """

    if pd.isna(value):
        return pd.NA

    # Extract all numeric GT IDs from the original value
    numbers = re.findall(r"\d+", str(value))

    if not numbers:
        return pd.NA

    # Multiple represented GT trees are separated with semicolons
    return ";".join(numbers)


# Prepare the Single-sensor candidate data

single_final = challenging_single_detection.copy()

single_final["represented_GT_IDs_final"] = (
    single_final["represented_GT_IDs"]
    .map(standardise_gt_ids)
)

single_final["GT_ID_final_clean"] = (
    single_final["GT_ID_final"]
    .map(standardise_gt_ids)
)

single_final["QC_notes_final"] = single_final["QC_notes"]


# Prepare the Dual-sensor candidate data

dual_final = challenging_dual_detection.copy()

dual_final["represented_GT_IDs_final"] = (
    dual_final["GT_ID"]
    .map(standardise_gt_ids)
)

# Keep a final GT ID only for valid one-to-one matches
dual_final["GT_ID_final_clean"] = np.where(
    dual_final["QC_class_final"].eq("validated_primary_match"),
    dual_final["represented_GT_IDs_final"],
    pd.NA,
)

dual_final["QC_notes_final"] = dual_final["notes"]


# Prepare the Hovermap candidate data

hovermap_final = challenging_hovermap_detection.copy()

hovermap_final["represented_GT_IDs_final"] = (
    hovermap_final["GT_ID"]
    .map(standardise_gt_ids)
)

# Keep a final GT ID only for valid one-to-one matches
hovermap_final["GT_ID_final_clean"] = np.where(
    hovermap_final["QC_class_final"].eq("validated_primary_match"),
    hovermap_final["represented_GT_IDs_final"],
    pd.NA,
)

hovermap_final["QC_notes_final"] = hovermap_final["notes"]


# Combine the three scanner-specific candidate tables

challenging_candidate_final = pd.concat(
    [
        single_final,
        dual_final,
        hovermap_final,
    ],
    ignore_index=True,
)


# Keep only the fields needed for the final candidate-level detection dataset
challenging_candidate_final = challenging_candidate_final[
    [
        "Terrain",
        "Scanner",
        "segment_id",
        "GT_ID_final_clean",
        "represented_GT_IDs_final",
        "source_QC_status",
        "QC_class_final",
        "GT_detection_credit",
        "include_in_attribute_accuracy_final",
        "is_genuine_unrecorded_tree",
        "is_undersegmented",
        "is_invalid_candidate",
        "QC_notes_final",
    ]
].copy()


# Rename the standardised fields to the names used in the final combined candidate dataset
challenging_candidate_final = challenging_candidate_final.rename(
    columns={
        "GT_ID_final_clean": "GT_ID_final",
        "represented_GT_IDs_final": "represented_GT_IDs",
        "QC_class_final": "QC_class",
        "include_in_attribute_accuracy_final":
            "include_in_attribute_accuracy",
        "QC_notes_final": "QC_notes",
    }
)


# Check the final Challenging candidate table

require(
    len(challenging_candidate_final) == 128,
    "Expected 128 Challenging RCT candidates."
)


# Each RCT segment should appear only once within each scanner
require(
    not challenging_candidate_final.duplicated(
        subset=["Terrain", "Scanner", "segment_id"]
    ).any(),
    "Duplicate Challenging candidate identifiers found."
)


# Every candidate should have a final QC classification
require(
    challenging_candidate_final["QC_class"].notna().all(),
    "At least one Challenging candidate has no final QC class."
)


print("Challenging candidate table created successfully.")
print("Rows:", len(challenging_candidate_final))

print("\nRows per scanner:")
print(
    challenging_candidate_final["Scanner"]
    .value_counts()
)

print("\nFinal QC classes:")
print(
    challenging_candidate_final["QC_class"]
    .value_counts()
)

Challenging candidate table created successfully.
Rows: 128

Rows per scanner:
Hovermap         46
Single sensor    43
Dual sensor      39
Name: Scanner, dtype: int64

Final QC classes:
validated_primary_match           57
genuine_unrecorded_tree           31
invalid_fragment                  17
undersegmented_GT_trees           12
invalid_non_tree_segment          10
genuine_unrecorded_fallen_tree     1
Name: QC_class, dtype: int64


In [50]:
# Put Easy and Intermediate into the same candidate format
# Give the Easy and Intermediate candidate tables the same structure as the Challenging table
# so that all three terrains can be combined into one candidate-level dataset
# GT_ID_final is kept only for valid one-to-one GT matches
# represented_GT_IDs records any GT tree associated with the candidate, including fragments linked to a reference tree

# Prepare the Easy candidate table
easy_candidate_final = easy_detection.copy()

easy_candidate_final["Terrain"] = "Easy"
easy_candidate_final["source_dataset"] = easy_candidate_final["dataset"]


# Keep the most specific original QC status available
easy_candidate_final["source_QC_status"] = (
    easy_candidate_final["manual_QC_class"]
    .combine_first(easy_candidate_final["candidate_status"])
)


# Record any GT tree associated with the candidate
easy_candidate_final["represented_GT_IDs"] = (
    easy_candidate_final["GT_ID_final"]
    .map(standardise_gt_ids)
)


# Keep GT_ID_final only for valid one-to-one GT matches
easy_candidate_final["GT_ID_final_clean"] = np.where(
    easy_candidate_final["QC_class"].eq("validated_primary_match"),
    easy_candidate_final["represented_GT_IDs"],
    pd.NA,
)


# Easy does not contain under-segmented candidates in the final QC
easy_candidate_final["is_undersegmented"] = False

# Split fragments are kept as invalid candidate-level detections
easy_candidate_final["is_invalid_candidate"] = (
    easy_candidate_final["QC_class"].eq("split_fragment")
)

easy_candidate_final["is_genuine_edge_truncated_tree"] = False

easy_candidate_final["QC_notes_final"] = (
    easy_candidate_final["manual_QC_notes"]
)


# Prepare the Intermediate candidate table

intermediate_candidate_final = intermediate_detection.copy()

intermediate_candidate_final["Terrain"] = "Intermediate"
intermediate_candidate_final["source_dataset"] = (
    intermediate_candidate_final["dataset"]
)


# Keep the most specific original QC status available
intermediate_candidate_final["source_QC_status"] = (
    intermediate_candidate_final["manual_QC_class"]
    .combine_first(intermediate_candidate_final["candidate_status"])
)


# Record any GT tree associated with the candidate
intermediate_candidate_final["represented_GT_IDs"] = (
    intermediate_candidate_final["GT_ID_final"]
    .map(standardise_gt_ids)
)


# Keep GT_ID_final only for valid one-to-one GT matches
intermediate_candidate_final["GT_ID_final_clean"] = np.where(
    intermediate_candidate_final["QC_class"].eq("validated_primary_match"),
    intermediate_candidate_final["represented_GT_IDs"],
    pd.NA,
)


intermediate_candidate_final["is_undersegmented"] = False
intermediate_candidate_final["is_invalid_candidate"] = False

intermediate_candidate_final["QC_notes_final"] = (
    intermediate_candidate_final["manual_QC_notes"]
)


# Define the columns that should have the same meaning across all three terrain-level candidate tables

common_candidate_columns = [
    "Terrain",
    "Scanner",
    "source_dataset",
    "segment_id",
    "GT_ID_final",
    "represented_GT_IDs",
    "source_QC_status",
    "QC_class",
    "GT_detection_credit",
    "include_in_attribute_accuracy",
    "is_genuine_unrecorded_tree",
    "is_genuine_edge_truncated_tree",
    "is_undersegmented",
    "is_invalid_candidate",
    "QC_notes",
]


# Temporarily rename the cleaned GT-ID and QC-note fields so they can replace the older working versions without ambiguity

easy_candidate_final = easy_candidate_final.rename(
    columns={
        "GT_ID_final_clean": "GT_ID_final_output",
        "QC_notes_final": "QC_notes_output",
    }
)

intermediate_candidate_final = intermediate_candidate_final.rename(
    columns={
        "GT_ID_final_clean": "GT_ID_final_output",
        "QC_notes_final": "QC_notes_output",
    }
)


# Replace the older working fields with the cleaned final versions

for df in [
    easy_candidate_final,
    intermediate_candidate_final,
]:

    df["GT_ID_final"] = df["GT_ID_final_output"]
    df["QC_notes"] = df["QC_notes_output"]

    df.drop(
        columns=[
            "GT_ID_final_output",
            "QC_notes_output",
        ],
        inplace=True,
    )


# Keep only the common final fields

easy_candidate_final = easy_candidate_final[
    common_candidate_columns
].copy()

intermediate_candidate_final = intermediate_candidate_final[
    common_candidate_columns
].copy()


# Add the remaining common fields to the Challenging table

challenging_candidate_final = challenging_candidate_final.copy()


# Recreate the original scanner dataset name so the same source_dataset field is available for all three terrains
challenging_candidate_final["source_dataset"] = (
    challenging_candidate_final["Scanner"]
    .map(
        {
            "Single sensor": "cht_jednoskener_nogrid",
            "Dual sensor": "cht_dvojskener_nogrid",
            "Hovermap": "cht_hovermap_nogrid",
        }
    )
)


challenging_candidate_final["is_genuine_edge_truncated_tree"] = False


challenging_candidate_final = challenging_candidate_final[
    common_candidate_columns
].copy()


print("Easy candidates:", len(easy_candidate_final))
print("Intermediate candidates:", len(intermediate_candidate_final))
print("Challenging candidates:", len(challenging_candidate_final))

Easy candidates: 31
Intermediate candidates: 72
Challenging candidates: 128


In [51]:
# Combine all RCT candidates
# This creates the complete candidate-level dataset used for the tree-detection analysis
# It keeps every RCT candidate, including valid GT matches, fragments, genuine unrecorded trees and under-segmented cases

rct_candidate_qc_master = pd.concat(
    [
        easy_candidate_final,
        intermediate_candidate_final,
        challenging_candidate_final,
    ],
    ignore_index=True,
)


# Check that the combined table contains the expected total number of candidates across all three terrains
require(
    len(rct_candidate_qc_master) == 231,
    "Expected 231 RCT candidates across all terrains."
)


# Each RCT segment should appear only once within each terrain and scanner combination
require(
    not rct_candidate_qc_master.duplicated(
        subset=["Terrain", "Scanner", "segment_id"]
    ).any(),
    "Duplicate Terrain + Scanner + segment_id combinations found."
)


# Every candidate should have a final QC classification
require(
    rct_candidate_qc_master["QC_class"].notna().all(),
    "At least one RCT candidate has no final QC classification."
)


# The older RCT_tree_id field should not appear in the cleaned candidate-level dataset
require(
    "RCT_tree_id" not in rct_candidate_qc_master.columns,
    "RCT_tree_id should not appear in the cleaned candidate dataset."
)


print("Combined RCT candidate table created successfully.")
print("Rows:", len(rct_candidate_qc_master))

print("\nCandidates by terrain and scanner:")
print(
    rct_candidate_qc_master
    .groupby(["Terrain", "Scanner"])
    .size()
)

print("\nFinal QC classes:")
print(
    rct_candidate_qc_master["QC_class"]
    .value_counts()
)

Combined RCT candidate table created successfully.
Rows: 231

Candidates by terrain and scanner:
Terrain       Scanner      
Challenging   Dual sensor      39
              Hovermap         46
              Single sensor    43
Easy          Dual sensor      10
              Hovermap         11
              Single sensor    10
Intermediate  Dual sensor      24
              Hovermap         24
              Single sensor    24
dtype: int64

Final QC classes:
validated_primary_match           150
genuine_unrecorded_tree            37
invalid_fragment                   17
undersegmented_GT_trees            12
invalid_non_tree_segment           10
genuine_edge_truncated_tree         3
split_fragment                      1
genuine_unrecorded_fallen_tree      1
Name: QC_class, dtype: int64


In [52]:
# Save the final RCT candidate-level QC dataset
# This file contains every RCT candidate used in the detection analysis, including:
# 1. valid GT matches
# 2. genuine unrecorded trees
# 3. fragments and under-segmented cases
# The original source files remain unchanged

RCT_CANDIDATE_QC_FILE = (
    RCT_PROCESSED_DIR / "rct_candidate_qc_master.csv"
)


rct_candidate_qc_master.to_csv(
    RCT_CANDIDATE_QC_FILE,
    index=False,
)


print(
    "Saved:",
    RCT_CANDIDATE_QC_FILE.relative_to(PROJECT_DIR)
)


# Read the saved file back in as a final export check
# This confirms that it can be loaded independently and still contains the expected rows and columns

candidate_export_check = pd.read_csv(
    RCT_CANDIDATE_QC_FILE
)


require(
    len(candidate_export_check) == 231,
    "The exported candidate QC file does not contain 231 rows."
)


require(
    candidate_export_check.columns.tolist()
    == rct_candidate_qc_master.columns.tolist(),
    "The exported candidate QC file has unexpected columns."
)


print(
    "Export check passed:",
    len(candidate_export_check),
    "rows and",
    len(candidate_export_check.columns),
    "columns."
)

Saved: data/processed/rct/rct_candidate_qc_master.csv
Export check passed: 231 rows and 15 columns.


In [53]:
# Cross-check the complete candidate and measurement datasets
# Every candidate retained for attribute accuracy should also appear in the final before/after measurement dataset
# This confirms that both final RCT datasets use exactly the same attribute-accuracy sample

candidate_measurement_keys = set(
    rct_candidate_qc_master.loc[
        rct_candidate_qc_master["include_in_attribute_accuracy"],
        ["Terrain", "Scanner", "segment_id"]
    ].itertuples(index=False, name=None)
)


measurement_keys = set(
    rct_before_after[
        ["Terrain", "Scanner", "segment_id"]
    ].itertuples(index=False, name=None)
)


require(
    candidate_measurement_keys == measurement_keys,
    "The final RCT candidate and measurement datasets do not "
    "contain the same attribute-accuracy sample."
)


print(
    "Final RCT cross-check passed:",
    len(measurement_keys),
    "attribute-accuracy records are present in both datasets."
)

Final RCT cross-check passed: 150 attribute-accuracy records are present in both datasets.


## RCT tree-detection metrics

Tree-detection performance is calculated separately for each terrain and scanner.

Completeness is the proportion of field-reference trees recovered as distinct individual RCT trees.

Strict correctness uses all RCT candidates in the denominator, including genuine trees that were not recorded in the field-reference inventory.

Adjusted correctness excludes manually verified genuine unreferenced trees from the denominator, but continues to penalise fragments, non-tree segments and under-segmented candidates.

In [54]:
# Calculate the final RCT tree-detection metrics
# The detection metrics are calculated directly from the complete candidate-level QC table
# Genuine trees that are missing from the field inventory are keptin the strict-correctness denominator
# but removed from the adjusted correctness calculation
# Invalid segmentation outputs remain penalised in both cases

GT_TREE_COUNTS = {
    "Easy": 9,
    "Intermediate": 22,
    "Challenging": 27,
}


genuine_unreferenced_classes = {
    "genuine_unrecorded_tree",
    "genuine_edge_truncated_tree",
    "genuine_unrecorded_fallen_tree",
}


invalid_classes = {
    "invalid_fragment",
    "split_fragment",
    "invalid_non_tree_segment",
    "undersegmented_GT_trees",
}


# Check that all QC classes belong to one of the groups used in the detection-correctness calculations

expected_classes = (
    {"validated_primary_match"}
    | genuine_unreferenced_classes
    | invalid_classes
)

observed_classes = set(
    rct_candidate_qc_master["QC_class"].dropna().unique()
)


require(
    observed_classes == expected_classes,
    "Unexpected or missing QC classes found before calculating "
    "the detection metrics."
)


def summarise_rct_detection(group):
    """Count the main candidate classes for one terrain and scanner."""

    qc = group["QC_class"]

    # Candidates representing a valid one-to-one GT tree match
    matched = qc.eq("validated_primary_match").sum()

    # Genuine trees that are present in the scan but absent from the reference inventory
    genuine_unreferenced = qc.isin(
        genuine_unreferenced_classes
    ).sum()

    # Segmentation errors and other invalid candidate outputs
    invalid = qc.isin(
        invalid_classes
    ).sum()

    total = len(group)

    return pd.Series(
        {
            "Total_candidates": total,
            "Matched_GT": matched,
            "Genuine_unreferenced": genuine_unreferenced,
            "Invalid_candidates": invalid,
        }
    )


# Count the different candidate types for each terrain and scanner

rct_detection_metrics = (
    rct_candidate_qc_master
    .groupby(
        ["Terrain", "Scanner"],
        observed=True,
    )
    .apply(summarise_rct_detection)
    .reset_index()
)


# Add the number of field-reference trees present in each terrain

rct_detection_metrics["GT_trees"] = (
    rct_detection_metrics["Terrain"]
    .map(GT_TREE_COUNTS)
)


# Sum the individual-tree detection credit assigned during QC
# Under the final detection rule, only valid one-to-one GT matches receive one unit of detection credit

individual_detection = (
    rct_candidate_qc_master
    .groupby(
        ["Terrain", "Scanner"],
        observed=True,
    )["GT_detection_credit"]
    .sum()
    .rename("Individually_detected")
    .reset_index()
)


rct_detection_metrics = rct_detection_metrics.merge(
    individual_detection,
    on=["Terrain", "Scanner"],
    how="left",
    validate="one_to_one",
)


# Calculate completeness as the percentage of reference GT trees recovered as separate individual RCT trees

rct_detection_metrics["Completeness_pct"] = (
    100
    * rct_detection_metrics["Individually_detected"]
    / rct_detection_metrics["GT_trees"]
)


# Calculate strict correctness
# Every RCT candidate remains in the denominator, including genuine trees that were not recorded in the reference inventory

rct_detection_metrics["Strict_correctness_pct"] = (
    100
    * rct_detection_metrics["Matched_GT"]
    / rct_detection_metrics["Total_candidates"]
)


# Calculate adjusted correctness
# Genuine unreferenced trees are removed from the denominator because they are real trees
# Invalid candidates, including under-segmented cases, remain

rct_detection_metrics["Adjusted_correctness_pct"] = (
    100
    * rct_detection_metrics["Matched_GT"]
    /
    (
        rct_detection_metrics["Matched_GT"]
        + rct_detection_metrics["Invalid_candidates"]
    )
)


# Check that the candidate classes account for every RCT candidate.

require(
    (
        rct_detection_metrics["Total_candidates"]
        ==
        (
            rct_detection_metrics["Matched_GT"]
            + rct_detection_metrics["Genuine_unreferenced"]
            + rct_detection_metrics["Invalid_candidates"]
        )
    ).all(),
    "Candidate classes do not sum to the total candidate count."
)


# Matched GT candidates should equal the total individual-tree detection credit under the final detection definition

require(
    (
        rct_detection_metrics["Matched_GT"]
        == rct_detection_metrics["Individually_detected"]
    ).all(),
    "Matched GT candidates and individual detections do not agree."
)


# Round the percentages for the final displayed table while leaving the underlying candidate counts unchanged

percentage_columns = [
    "Completeness_pct",
    "Strict_correctness_pct",
    "Adjusted_correctness_pct",
]

rct_detection_metrics[percentage_columns] = (
    rct_detection_metrics[percentage_columns]
    .round(1)
)


# Keep terrain and scanner results in the order used throughout the dissertation

rct_detection_metrics["Terrain"] = pd.Categorical(
    rct_detection_metrics["Terrain"],
    categories=TERRAINS,
    ordered=True,
)

rct_detection_metrics["Scanner"] = pd.Categorical(
    rct_detection_metrics["Scanner"],
    categories=SCANNERS,
    ordered=True,
)


rct_detection_metrics = (
    rct_detection_metrics
    .sort_values(["Terrain", "Scanner"])
    .reset_index(drop=True)
)


rct_detection_metrics[
    [
        "Terrain",
        "Scanner",
        "GT_trees",
        "Individually_detected",
        "Total_candidates",
        "Matched_GT",
        "Genuine_unreferenced",
        "Invalid_candidates",
        "Completeness_pct",
        "Strict_correctness_pct",
        "Adjusted_correctness_pct",
    ]
]

,Terrain,Scanner,GT_trees,Individually_detected,Total_candidates,Matched_GT,Genuine_unreferenced,Invalid_candidates,Completeness_pct,Strict_correctness_pct,Adjusted_correctness_pct
0,Easy,Single sensor,9,9,10,9,1,0,100.0,90.0,100.0
1,Easy,Dual sensor,9,9,10,9,1,0,100.0,90.0,100.0
2,Easy,Hovermap,9,9,11,9,1,1,100.0,81.8,90.0
3,Intermediate,Single sensor,22,22,24,22,2,0,100.0,91.7,100.0
4,Intermediate,Dual sensor,22,22,24,22,2,0,100.0,91.7,100.0
5,Intermediate,Hovermap,22,22,24,22,2,0,100.0,91.7,100.0
6,Challenging,Single sensor,27,19,43,19,9,15,70.4,44.2,55.9
7,Challenging,Dual sensor,27,19,39,19,11,9,70.4,48.7,67.9
8,Challenging,Hovermap,27,19,46,19,12,15,70.4,41.3,55.9


In [55]:
# Save the final RCT tree-detection summary
# These metrics are calculated directly from the cleaned candidate-level QC dataset

RCT_DETECTION_SUMMARY_FILE = (
    RCT_PROCESSED_DIR / "rct_detection_metrics.csv"
)


rct_detection_metrics.to_csv(
    RCT_DETECTION_SUMMARY_FILE,
    index=False,
)


print(
    "Saved:",
    RCT_DETECTION_SUMMARY_FILE.relative_to(PROJECT_DIR)
)

Saved: data/processed/rct/rct_detection_metrics.csv


In [56]:
# Read the saved detection summary back in as a final export check
# This confirms that the file can be loaded independently and contains the expected nine terrain-scanner combinations

rct_detection_export_check = pd.read_csv(
    RCT_DETECTION_SUMMARY_FILE
)


require(
    len(rct_detection_export_check) == 9,
    "Expected 9 terrain-scanner rows in the exported detection summary."
)


print(
    "Export check passed:",
    len(rct_detection_export_check),
    "rows and",
    len(rct_detection_export_check.columns),
    "columns."
)

Export check passed: 9 rows and 11 columns.


In [57]:
# Final RCT preparation check
# Confirm that all processed RCT datasets needed for the later analysis have been created successfully
# before moving on to the 3DFin workflow.

expected_rct_outputs = [
    RCT_PROCESSED_DIR / "rct_before_after_per_tree.csv",
    RCT_PROCESSED_DIR / "rct_candidate_qc_master.csv",
    RCT_PROCESSED_DIR / "rct_detection_metrics.csv",
]


for path in expected_rct_outputs:
    require(
        path.exists(),
        f"Missing expected RCT output: {path.name}"
    )
    print("OK", path.name)


print("\nRCT data preparation complete.")

OK rct_before_after_per_tree.csv
OK rct_candidate_qc_master.csv
OK rct_detection_metrics.csv

RCT data preparation complete.
